## RUL CPFL — Timestral congelado e heads supervisionados

Este notebook avalia uma cadeia causal específica:

**sensores observados → trajetórias futuras de sensores pelo Timestral congelado → head Ridge/ExtraTrees → RUL**

O Timestral é usado apenas como forecaster de sensores, sem ajuste de pesos e sem canal de RUL. A RUL é aprendida e avaliada pelos heads supervisionados. Não há um baseline de RUL zero-shot, nem repetição dos baselines standalone, nem controle sem forecast nesta notebook.

O protocolo de dados, limpeza, endpoints, grades, métricas e persistência é compartilhado com o notebook de baselines; as células genuinamente comuns são mantidas idênticas.


## RUL CPFL — Timestral congelado + heads supervisionados

Notebook em português que utiliza o módulo compartilhado `cer_rul.py` e testa uma hipótese específica:

1. os oito sensores **observados** até um endpoint `t` formam um contexto causal;
2. o **Timestral congelado** (sem fine-tuning) prevê trajetórias futuras desses sensores;
3. um head supervisionado **Ridge** ou **ExtraTrees** aprende a mapear atributos observados + atributos das trajetórias previstas para a RUL.

Não é RUL zero-shot. O Timestral não vê a RUL, não recebe um canal de RUL e não é treinado neste notebook. A RUL é prevista apenas pelos heads supervisionados. Também não há repetição dos baselines standalone (média/Ridge/SVR/ExtraTrees sobre atributos observados) nem um controle de “sem forecast” sendo treinado aqui.

## Como usar

**Pré-requisito:** antes de executar qualquer célula, envie `cer_rul.py` pelo painel de arquivos do Colab para `/content`. Um runtime novo perde o arquivo, então ele deve ser reenviado a cada runtime; pode ser o mesmo `.py` do repositório. Nenhum notebook baixa ou instala `cer_rul.py` automaticamente.

1. Execute em ordem. A configuração é a **única** célula de configuração do usuário.
2. O padrão é **smoke de dados**: `run_training=False`, `forecast_smoke=False`, `run_forecast=False`. Ele valida o CSV do dataset selecionado, audita a limpeza, monta EDA/endpoints/splits, imprime o preview de forecasts e ajustes, e exporta um ZIP — sem baixar pesos Timestral e sem treinar.
3. **Dataset:** `CONFIG["dataset"] = "gerador"` (padrão) ou `"gearbox"`. Cada execução usa **um** CSV `porcentagem-data`; os datasets são independentes e nunca são mesclados. Para um arquivo explícito, use `CONFIG["input"]["mode"] = "files"` e `csv_paths=[...]`; um CSV de outro dataset gera erro.
4. **Pré-processamento:** `CONFIG["preprocessing"] = {"method": ..., "add_missing_indicators": True}` é o único controle para `quick`. Métodos: `none`, `standard`, `robust`, `minmax`, `signed_log1p_standard`. `search.preprocessing_levels` é apenas o override da busca Optuna, escolhido no fold interno.
5. Para testar o Timestral real em poucos endpoints, sem treinar heads:

```python
CONFIG["model_specific"]["forecast_smoke"] = True
CONFIG["run_training"] = False
```

6. Para a execução científica completa (forecasts em todos os endpoints do dataset + 2 heads × rótulo × folds):

```python
CONFIG["run_training"] = True
CONFIG["model_specific"]["run_forecast"] = True
```

Depois **REEXECUTE TODAS AS CÉLULAS**. Gerador: 574 endpoints, 2 heads × 4 folds = 8 payloads. Gearbox: 822 endpoints, 2 heads × 3 folds = 6 payloads.

7. A saída persistente padrão é o Google Drive. O Drive é montado antes de qualquer `mkdir`/escrita de saída, inclusive quando a entrada é ZIP/CSV enviado. O modo local emite aviso explícito de efemeridade.
8. Instalação opcional do pacote Timestral (somente Colab): `model_specific.install_dependencies=True`. A URL é fixada no commit aprovado. Sem o pacote, ativar forecast produz erro acionável; o smoke de dados continua funcionando.

## Protocolo compartilhado

- Dataset selecionado: gerador (4 equipamentos `A=[1,2]`, `B=[3]`, `C=[4,5]`, `D=[6,7]`; 2.710 linhas brutas, 1.910 chaves limpas, 574 endpoints) ou gearbox (3 eventos `E1=[1..6]`, `E2=[7..12]`, `E3=[13..18]`; 2.946 linhas, 2.946 chaves limpas, 822 endpoints). Em ambos, a média de **todas** as linhas originais de RUL por `(Falha, Date)` e sensores idênticos; nenhum zero terminal é imposto.
- Um grupo de teste por fold (leave-one-group-out), um grupo de validação interna, os demais no treino interno; no gearbox, cada fold tem um evento de teste, um de validação, um de treino interno e dois no refit externo. Seleção apenas pela MAE macro por componente no fold interno; refit no treino externo e teste externo intocado.
- Endpoints após 30 registros de aquecimento, passo 3 e endpoint final. Bins regulares de 32 passos, ancorados à direita em `t`, `(left, right]`, último valor observado por sensor; ausente é NaN. Larguras 6/12/24 h, padrão 12 h. Sem interpolação, `ffill` ou valores futuros.
- `current` usa o último valor observado por sensor dentro da janela, contagens e frações de ausência. `summary` acrescenta média, desvio e inclinação por **dia real** na mesma grade. Nenhum valor bruto de bin entra na matriz tabular.
- Ids, `Falha`, `Date`, RUL, fase, FPT e instante de falha nunca entram como atributos de inferência.
- FPTs do dataset selecionado servem apenas para relatório de fase e pesos opcionais de treino. No gerador são herdadas (`1/2=2024-07-25`, `3=2024-03-15`, `4/5=2023-12-10`, `6/7=2024-07-22`); no gearbox são inferidas da rotulagem linear (`E1=2023-06-14`, `E2=2022-11-01`, `E3=2023-08-13`). Em ambos não foram verificadas fisicamente de forma independente.
- Métricas: MAE, RMSE e viés (`pred - real`), por componente/grupo/fase e macro igual entre os grupos do dataset; dias nativos ou pontos percentuais. Sem MAPE, clamp, escala pelo ciclo de vida do teste ou ranqueamento entre rótulos.

## Forecast e heads

- Contexto Timestral: `(B, 8, 32)` com os oito sensores crus positivos, NaN para ausência; nenhuma escala externa obrigatória (o modelo faz sua escala interna). Sem RUL observada ou totalmente ausente e sem covariáveis FPT conhecidas no futuro.
- Chamada congelada: `TimestralForecaster.from_pretrained(model_repo, revision=..., device=...)` e `.forecast(context, prediction_length=steps, symmetric_averaging=True, clip_nonnegative=True)`. Saídas `.quantile(0.5)`, `.quantile(0.1)` e `.quantile(0.9)`, `(B, 8, steps)`.
- Atributos de forecast por sensor: último mediano, média dos medianos, mudança em relação ao último observado, inclinação por dia de forecast e largura média `q90-q10`. Eles são adicionados aos atributos observados; as trajetórias futuras brutas não entram na matriz tabular.
- Heads: Ridge e ExtraTrees, com seleção/refit/teste **por head**, sem vencedor global. Cada head tem sua própria configuração escolhida no fold interno e seu teste externo.
- O cache é **somente de sensores/atributos de forecast** e sua chave contém dataset, protocolo e modelo, mas nenhum rótulo nem hash de alvo. Assim os forecasts congelados de um dataset são reutilizados pelos folds e tentativas daquele dataset. A incerteza dos sensores não é um intervalo de confiança da RUL, e nenhum limiar é tratado como falha.

## Limitações

- Gerador: quatro equipamentos. Gearbox: três eventos (não quatro máquinas). Em ambos há um único grupo de validação interna por fold: a seleção interna tem baixa resolução.
- O cenário é generalização para um grupo não visto, não previsão do futuro do mesmo equipamento ou evento.
- O Timestral é congelado e pode errar as trajetórias dos sensores; o head supervisionado não corrige essa incerteza nem a transforma em intervalo de RUL.
- Não há controle “sem forecast” nesta notebook; portanto uma diferença para os baselines standalone não isola o efeito do Timestral. A comparação opcional com exports do notebook de baselines é apenas descritiva, exige identidades compatíveis (dataset, protocolo, endpoints, splits) e nunca seleciona modelo pelo teste importado. Se `comparison.enabled=True` mas não há métricas Timestral de treino completo (por exemplo, smoke de dados), a comparação é **adiada** com status `deferred_no_current_metrics`; nenhum score importado é lido ou usado.
- SQLite em Drive não é seguro para múltiplos escritores. Use um runtime por vez.
- A execução científica real no Colab não foi realizada na criação deste notebook; os testes locais foram estruturais/sintéticos e não usaram os CSVs para ajustar nada.


In [ ]:
# =============================================================================
# CÉLULA ÚNICA DE CONFIGURAÇÃO — TIMESTRAL
# Altere apenas o que estiver indicado. Depois de qualquer mudança, reexecute
# TODAS as células (Runtime > Restart and run all). EFFECTIVE_CONFIG/RUN_ID são
# derivados desta configuração em células posteriores.
# =============================================================================

CONFIG = {
    # Smoke padrão: dados, limpeza, endpoints, EDA e previews. Nenhum modelo
    # Timestral é baixado/executado e nenhum head é ajustado.
    "run_training": False,

    # Identidade de código desta pipeline (entra no RUN_ID).
    "code_revision": "rul-cpfl-timestral-v2",
    "seed_base": 20261015,

    # Dataset ativo: "gerador" (padrão, 4 equipamentos) ou "gearbox" (3 eventos).
    # Um único CSV porcentagem-data por execução; os datasets nunca são mesclados.
    "dataset": "gerador",

    # Pré-processamento de quick e ablação (único controle). method: none |
    # standard | robust | minmax | signed_log1p_standard. add_missing_indicators
    # deve ser True ou False (bool). A busca Optuna usa search.preprocessing_levels.
    "preprocessing": {"method": "standard", "add_missing_indicators": True},

    "input": {
        # "auto": tenta input_dir, depois caminhos explícitos, depois upload.
        # "drive_dir": exige input_dir. "files": usa somente zip_paths/csv_paths.
        "mode": "auto",
        "drive_dir": "/content/drive/MyDrive/Grupo2_Prognóstico/Resultados_preprocessamento_CPFL",
        "zip_paths": [],
        "csv_paths": [],
        "allow_colab_upload": True,
    },

    "output": {
        # A saída persistente padrão é o Drive. "local" avisa explicitamente
        # sobre efemeridade e é destinado apenas a smoke fora do Colab.
        "mode": "drive",
        "drive_subdir": "RUL_CPFL_timestral",
        "local_dir": "/tmp/opencode/rul_cpfl_timestral",
        "download_zip": True,
        "local_only_acknowledged": False,
        "install_dependencies": False,
    },

    # Presets pequenos para os DOIS heads supervisionados. Não há média,
    # SVR nem uma repetição dos baselines standalone nesta notebook.
    # O pré-processamento NÃO aparece nos presets: vem de CONFIG["preprocessing"].
    "quick": {
        "presets": {
            "ridge": [
                {"alpha": 1.0, "horizon_days": 3,
                 "bin_hours": 12, "representation": "summary",
                 "timing": False, "weights": "none"},
                {"alpha": 10.0, "horizon_days": 3,
                 "bin_hours": 12, "representation": "current",
                 "timing": False, "weights": "component"},
            ],
            "extra_trees": [
                {"max_depth": 8, "min_samples_leaf": 2, "max_features": "sqrt",
                 "horizon_days": 3, "bin_hours": 12,
                 "representation": "summary",
                 "timing": False, "weights": "none"},
                {"max_depth": 10, "min_samples_leaf": 1, "max_features": 0.5,
                 "horizon_days": 3, "bin_hours": 12,
                 "representation": "current",
                 "timing": False, "weights": "component"},
            ],
        },
    },

    # Optuna opcional: 15    # Optuna opcional: 15 tentativas TOTAIS por estudo
    # (head × rótulo × fold externo). A grade/horizonte também é escolhida
    # somente no fold interno. Ao retomar, valida a assinatura e roda apenas
    # as tentativas que faltam; nunca adiciona outras 15.
    "search": {
        "enabled": False,
        "n_trials": 15,
        "sampler_seed": 20261015,
        "families": ["ridge", "extra_trees"],
        "bin_hours_levels": [6, 12, 24],
        "horizon_days_levels": [1, 3, 7],
        "representation_levels": ["current", "summary"],
        # Override só da busca Optuna (não altera CONFIG["preprocessing"]).
        "preprocessing_levels": ["none", "standard", "robust", "minmax",
                                 "signed_log1p_standard"],
        "timing_levels": [False, True],
        "weights_levels": ["none", "component", "component_phase"],
        "spaces": {
            "ridge": {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
            "extra_trees": {
                "max_depth": [4, 6, 8, 10, 12],
                "min_samples_leaf": [1, 2, 4],
                "max_features": ["sqrt", 0.5, 1.0],
            },
        },
    },

    # Ablação opcional de UM eixo, com parâmetros/horizonte/grade/seed fixos.
    # Todos os níveis são refeitos no teste externo; não há vencedor externo.
    "ablation": {
        "enabled": False,
        "axis": "representation",
        "levels": ["current", "summary"],
        "base_family": "ridge",
        "base_params": {"alpha": 1.0},
        "base_horizon_days": 3,
        "base_bin_hours": 12,
        "base_representation": "summary",
        "base_timing": False,
        "base_weights": "none",
    },

    # Configuração congelada do sensor forecaster. O alvo RUL nunca é um canal
    # de contexto e nunca é usado para escolher features de forecast.
    "model_specific": {
        "pipeline_id": "timestral_forecast_plus_ridge_extratrees",
        "model_repo": "mistralai/Timestral-28M",
        "model_revision": "49b26409ac87b5935539a5d8ffe7f397b720db47",
        "package_commit": "a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff",
        "package_url": "git+https://github.com/mistralai/timestral.git@a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff",
        "device": "cpu",                 # "cpu" | "cuda" | "auto"
        "context_length": 32,
        "default_horizon_days": 3,
        "batch_size": 8,
        "symmetric_averaging": True,
        "clip_nonnegative": True,
        "forecast_feature_version": "timestral-forecast-features-v1",
        "cache_subdir": "timestral",
        # Explícito e desligado por padrão: não baixa pesos no smoke de dados.
        "run_forecast": False,
        "forecast_smoke": False,
        "install_dependencies": False,
    },

    # Comparação opcional, sem ajuste e sem escolha de modelo. Só é lida se
    # enabled=True e exige identidades de dados/protocolo/endpoints/splits/unidades.
    "comparison": {
        "enabled": False,
        "baseline_run_dir": "",
    },
}

# Identidades de pipeline usadas em arquivos, payloads e métricas. Os heads
# internos são "ridge" e "extra_trees"; o prefixo os distingue dos baselines.
MODEL_FAMILIES = ("timestral_ridge", "timestral_extra_trees")
PIPELINE_BY_HEAD = {"ridge": "timestral_ridge",
                    "extra_trees": "timestral_extra_trees"}
HEAD_BY_PIPELINE = {value: key for key, value in PIPELINE_BY_HEAD.items()}


def validate_timestral_config():
    """Falha cedo se listas obrigatórias forem silenciosamente reduzidas."""
    heads = set(PIPELINE_BY_HEAD)
    if set(MODEL_FAMILIES) != set(PIPELINE_BY_HEAD.values()):
        raise ValueError("MODEL_FAMILIES deve conter exatamente os dois heads.")
    if set(CONFIG["quick"]["presets"]) != heads:
        raise ValueError(
            "quick.presets deve conter exatamente ridge e extra_trees; "
            f"recebido={sorted(CONFIG['quick']['presets'])}."
        )
    if set(CONFIG["search"]["families"]) != heads:
        raise ValueError(
            "search.families deve conter exatamente ridge e extra_trees; "
            "não é permitido ignorar um head."
        )
    if set(CONFIG["search"]["spaces"]) != heads:
        raise ValueError("search.spaces deve conter exatamente os dois heads.")
    if CONFIG["ablation"]["base_family"] not in heads:
        raise ValueError("ablation.base_family deve ser ridge ou extra_trees.")
    if int(CONFIG["model_specific"]["context_length"]) != 32:
        raise ValueError("model_specific.context_length deve ser 32.")
    package_url = str(CONFIG["model_specific"]["package_url"])
    package_commit = str(CONFIG["model_specific"]["package_commit"])
    if not package_url.endswith("@" + package_commit):
        raise ValueError("package_url deve fixar o mesmo commit de package_commit.")
    if not str(CONFIG["model_specific"]["model_repo"]).strip():
        raise ValueError("model_specific.model_repo não pode ser vazio.")
    if not str(CONFIG["model_specific"]["model_revision"]).strip():
        raise ValueError("model_specific.model_revision não pode ser vazio.")
    if CONFIG["run_training"] and not CONFIG["model_specific"]["run_forecast"]:
        raise ValueError(
            "run_training=True exige model_specific.run_forecast=True; "
            "não há fallback silencioso nem mock."
        )
    if (CONFIG["comparison"]["enabled"] and CONFIG["run_training"]
            and not CONFIG["comparison"]["baseline_run_dir"]):
        raise ValueError(
            "comparison.enabled com run_training=True exige "
            "comparison.baseline_run_dir; em smoke de dados ela fica adiada."
        )


validate_timestral_config()


def _describe_run_switch():
    print("run_training:", CONFIG["run_training"])
    print("forecast_smoke:", CONFIG["model_specific"]["forecast_smoke"],
          "| run_forecast:", CONFIG["model_specific"]["run_forecast"])
    if not CONFIG["run_training"] and not CONFIG["model_specific"]["forecast_smoke"]:
        print("SMOKE DE DADOS: nenhum peso Timestral será baixado e nenhum head será ajustado.")
    elif CONFIG["model_specific"]["forecast_smoke"] and not CONFIG["run_training"]:
        print("SMOKE DE FORECAST EXPLÍCITO: baixa/usa Timestral em poucos endpoints, sem treinar heads.")
    elif CONFIG["run_training"] and not CONFIG["model_specific"]["run_forecast"]:
        print("AVISO: run_training=True exige model_specific.run_forecast=True.")
    print("Depois de alterar a configuração, reexecute TODAS as células.")


_describe_run_switch()


In [ ]:
# =============================================================================
# SEÇÃO 1 — AMBIENTE, DRIVE E DIRETÓRIOS
# Nenhum mkdir/escrita de saída ocorre antes da montagem do Drive.
# =============================================================================

import os
import sys
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

OUTPUT_MODE = CONFIG["output"]["mode"]
DRIVE_MOUNTED = False

if OUTPUT_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError(
            "output.mode='drive' exige Google Colab. Use mode='local' apenas "
            "para um smoke local e leia o aviso de efemeridade."
        )
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / CONFIG["output"]["drive_subdir"]
elif OUTPUT_MODE == "local":
    OUTPUT_ROOT = Path(CONFIG["output"]["local_dir"])
    if IN_COLAB:
        print("AVISO: saída local no Colab é efêmera e pode desaparecer no restart.")
    else:
        print("AVISO EXPLÍCITO: modo local/efêmero; nada será persistido no Drive.")
        if not CONFIG["output"]["local_only_acknowledged"]:
            print("Defina output.local_only_acknowledged=True se esta for a intenção.")
else:
    raise ValueError("output.mode deve ser 'drive' ou 'local'.")

# Primeiro efeito de escrita só depois da montagem (ou do aviso local explícito).
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_DIR = OUTPUT_ROOT / "cache"
STUDIES_DIR = CACHE_DIR / "studies"
INPUT_CACHE_DIR = CACHE_DIR / "inputs"
FEATURE_CACHE_DIR = CACHE_DIR / "features"
RESULTS_DIR = OUTPUT_ROOT / "results"
EXPORTS_DIR = OUTPUT_ROOT / "exports"
for _directory in (CACHE_DIR, STUDIES_DIR, INPUT_CACHE_DIR,
                   FEATURE_CACHE_DIR, RESULTS_DIR, EXPORTS_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

if CONFIG["output"]["install_dependencies"]:
    if not IN_COLAB:
        raise RuntimeError("Instalação opcional é permitida somente no Colab.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "pandas", "numpy", "scikit-learn", "matplotlib", "optuna"],
        check=True,
    )

# Núcleo compartilhado. No Colab, envie cer_rul.py para /content antes de
# executar o notebook. Localmente, inicie o Jupyter na raiz do repositório
# que contém cer_rul.py. Não há upload automático nem download do módulo.
try:
    import cer_rul as _cer_rul
except ModuleNotFoundError as exc:
    if exc.name != "cer_rul":
        raise
    raise ModuleNotFoundError(
        "cer_rul.py não encontrado. No Colab, envie cer_rul.py para /content "
        "(painel de arquivos ou google.colab.files.upload()). Localmente, "
        "inicie o Jupyter na raiz do repositório que contém cer_rul.py."
    ) from exc

RUNTIME_STARTED_UTC = datetime.now(timezone.utc).isoformat()
print("Runtime:", "Colab" if IN_COLAB else "fora do Colab")
print("Saída:", OUTPUT_ROOT)
print("Início UTC:", RUNTIME_STARTED_UTC)


In [ ]:
# =============================================================================
# INSTALAÇÃO OPCIONAL DO PACOTE TIMESTRAL (SOMENTE COLAB)
# Não baixa pesos. A URL e o commit são os metadados caros registrados no RUN_ID.
# =============================================================================

if CONFIG["model_specific"]["install_dependencies"]:
    if not IN_COLAB:
        raise RuntimeError(
            "A instalação opcional do Timestral é permitida somente no Colab."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         CONFIG["model_specific"]["package_url"]],
        check=True,
    )
    print("Timestral instalado a partir de:",
          CONFIG["model_specific"]["package_url"])
else:
    print("Instalação do Timestral desligada "
          "(model_specific.install_dependencies=False).")


In [ ]:
# =============================================================================
# SEÇÃO 1 (continuação) — BIBLIOTECAS CIENTÍFICAS
# Nenhum dado é ajustado nesta célula.
# =============================================================================

import importlib.metadata as importlib_metadata
import json
import math
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


def _installed_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None


RUNTIME_VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": _installed_version("scikit-learn"),
    "matplotlib": matplotlib.__version__,
    "optuna": _installed_version("optuna"),
}
print(RUNTIME_VERSIONS)


In [ ]:
# =============================================================================
# VERSÕES E PROVENIÊNCIA EFETIVA DO PACOTE TIMESTRAL
# Nenhum torch ou timestral é importado nesta célula; apenas metadados.
# =============================================================================

def _metadata_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None


def _version_tuple(value):
    if value is None:
        return None
    parts = []
    for token in str(value).split("."):
        digits = "".join(character for character in token if character.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


def installed_direct_url(distribution="timestral"):
    try:
        distribution_metadata = importlib_metadata.distribution(distribution)
    except Exception:
        return None
    try:
        raw = distribution_metadata.read_text("direct_url.json")
    except Exception:
        return None
    if not raw:
        return None
    try:
        return json.loads(raw)
    except Exception:
        return None


def audit_timestral_source(direct_url=None, installed_version=None,
                           torch_version=None, python_version=None):
    """Audita a origem instalada; não importa nem baixa o pacote."""
    model_cfg = CONFIG["model_specific"]
    configured_commit = str(model_cfg["package_commit"])
    configured_url = str(model_cfg["package_url"])
    if not configured_url.endswith("@" + configured_commit):
        raise ValueError(
            "package_url e package_commit não são consistentes: a URL deve "
            "terminar em '@<commit>'."
        )
    if direct_url is None:
        direct_url = installed_direct_url("timestral")
    if installed_version is None:
        installed_version = _metadata_version("timestral")
    if torch_version is None:
        torch_version = _metadata_version("torch")
    if python_version is None:
        python_version = sys.version_info[:3]

    actual_commit = None
    actual_url = None
    source_status = "missing_package"
    if direct_url is not None:
        actual_url = str(direct_url.get("url", ""))
        vcs_info = direct_url.get("vcs_info") or {}
        actual_commit = (vcs_info.get("commit_id")
                         or vcs_info.get("requested_revision"))
        if actual_commit is None:
            source_status = "missing_vcs_commit"
        elif actual_commit != configured_commit:
            source_status = "commit_mismatch"
        elif "mistralai/timestral" not in actual_url:
            source_status = "url_mismatch"
        else:
            source_status = "verified"

    torch_tuple = _version_tuple(torch_version)
    python_tuple = tuple(int(part) for part in python_version)
    return {
        "configured_package_url": configured_url,
        "configured_package_commit": configured_commit,
        "installed_version": installed_version,
        "installed_url": actual_url,
        "verified_package_commit": actual_commit,
        "source_status": source_status,
        "torch_version": torch_version,
        "python_version": ".".join(str(part) for part in python_tuple),
        "torch_ge_2_4": bool(torch_tuple is not None and torch_tuple >= (2, 4)),
        "python_ge_3_10": bool(python_tuple >= (3, 10)),
    }


def validate_timestral_runtime():
    audit = audit_timestral_source()
    if not audit["python_ge_3_10"]:
        raise RuntimeError(
            f"Timestral exige Python >= 3.10; runtime={audit['python_version']}."
        )
    if not audit["torch_ge_2_4"]:
        raise RuntimeError(
            "Timestral exige torch >= 2.4; "
            f"instalado={audit['torch_version']!r}."
        )
    if audit["source_status"] != "verified":
        raise RuntimeError(
            "A origem instalada do Timestral não corresponde ao commit aprovado. "
            f"status={audit['source_status']}; "
            f"configurado={audit['configured_package_commit']}; "
            f"verificado={audit['verified_package_commit']}. "
            "Ative model_specific.install_dependencies=True no Colab para "
            "instalar a URL fixada, ou instale exatamente o commit aprovado."
        )
    return audit


TIMESTRAL_SOURCE_AUDIT = audit_timestral_source()
TIMESTRAL_VERSIONS = {
    "torch": _metadata_version("torch"),
    "timestral": _metadata_version("timestral"),
    "model_repo": CONFIG["model_specific"]["model_repo"],
    "weights_revision": CONFIG["model_specific"]["model_revision"],
    "configured_package_commit": CONFIG["model_specific"]["package_commit"],
    "configured_package_url": CONFIG["model_specific"]["package_url"],
    "verified_package_commit": TIMESTRAL_SOURCE_AUDIT["verified_package_commit"],
    "source_status": TIMESTRAL_SOURCE_AUDIT["source_status"],
    "installed_url": TIMESTRAL_SOURCE_AUDIT["installed_url"],
    "dtype": "float32",
    "device_config": CONFIG["model_specific"]["device"],
    "python": TIMESTRAL_SOURCE_AUDIT["python_version"],
    "torch_ge_2_4": TIMESTRAL_SOURCE_AUDIT["torch_ge_2_4"],
    "python_ge_3_10": TIMESTRAL_SOURCE_AUDIT["python_ge_3_10"],
}
print("Auditoria da origem Timestral:", TIMESTRAL_VERSIONS)


In [ ]:
# =============================================================================
# SEÇÃO 2 — ESQUEMA, PROTOCOLO E CONSTANTES COMPARTILHADAS
# Tudo nesta célula é comum a qualquer notebook standalone que reutilize o
# mesmo protocolo causal (daí os IDs "common-*" das células no .ipynb).
# O dataset selecionado em CONFIG["dataset"] define colunas, grupos, IDs, FPT,
# rótulo e expectativas. Gerador e gearbox são CSVs independentes: nunca há merge.
# =============================================================================

DATASET_CHOICES = ("gerador", "gearbox")
PREPROCESS_METHOD_CHOICES = ("none", "standard", "robust", "minmax",
                             "signed_log1p_standard")


def validate_notebook_config(config):
    """Valida dataset e pré-processamento cedo, sem converter bool de texto."""
    if config.get("dataset") not in DATASET_CHOICES:
        raise ValueError(
            f"CONFIG['dataset'] deve ser um de {DATASET_CHOICES}; "
            f"recebido {config.get('dataset')!r}."
        )
    preprocessing = config.get("preprocessing")
    if (not isinstance(preprocessing, dict)
            or set(preprocessing) != {"method", "add_missing_indicators"}):
        raise ValueError(
            "CONFIG['preprocessing'] deve conter exatamente 'method' e "
            "'add_missing_indicators'."
        )
    if preprocessing["method"] not in PREPROCESS_METHOD_CHOICES:
        raise ValueError(
            f"CONFIG['preprocessing']['method'] deve ser um de "
            f"{PREPROCESS_METHOD_CHOICES}; recebido {preprocessing['method']!r}."
        )
    indicators = preprocessing["add_missing_indicators"]
    if not isinstance(indicators, bool):
        raise ValueError(
            "CONFIG['preprocessing']['add_missing_indicators'] deve ser True ou "
            f"False (bool); recebido {indicators!r}. Não há conversão implícita."
        )
    return {"method": preprocessing["method"],
            "add_missing_indicators": indicators}


PREPROCESSING_CONFIG = validate_notebook_config(CONFIG)
DATASET_PROFILE = _cer_rul.get_dataset_profile(CONFIG["dataset"])
DATASET_ID = DATASET_PROFILE["dataset_id"]
UNIT_KIND = DATASET_PROFILE["unit_kind"]
UNIT_LABEL = {"equipment": "equipamento", "event": "evento"}[UNIT_KIND]
UNIT_LABEL_PLURAL = {"equipment": "equipamentos", "event": "eventos"}[UNIT_KIND]

PROTOCOL = {
    "name": "RUL CPFL baselines causais",
    "revision": "1.1.0",
    "dataset_id": DATASET_ID,
    "unit_kind": UNIT_KIND,
    "warmup_records": 30,
    "endpoint_stride": 3,
    "context_bins": 32,
    "default_bin_hours": 12,
    "bin_hours_candidates": (6, 12, 24),
    "equipment_groups": {name: tuple(ids) for name, ids
                         in DATASET_PROFILE["equipment_groups"].items()},
    "measurement_by_id": {int(fault_id): name for fault_id, name
                          in DATASET_PROFILE["measurement_by_id"].items()},
    "selection_metric": "inner_MACRO_COMPONENT_MAE_ONLY",
    "phase_rule": "saudavel se Date < FPT; pos_FPT se Date >= FPT",
    "fpt_annotations": {int(fault_id): date for fault_id, date
                        in DATASET_PROFILE["fpt_annotations"].items()},
    "fpt_note": DATASET_PROFILE["fpt_note"],
    "labels": DATASET_PROFILE["labels"],
}

LABEL_ORDER = list(PROTOCOL["labels"].keys())
EQUIPMENT_ORDER = list(PROTOCOL["equipment_groups"].keys())
ID_TO_EQUIPMENT = {
    int(equipment_id): equipment_name
    for equipment_name, equipment_ids in PROTOCOL["equipment_groups"].items()
    for equipment_id in equipment_ids
}
FAULT_IDS = tuple(sorted(ID_TO_EQUIPMENT))

SENSOR_COLUMNS = tuple(DATASET_PROFILE["sensor_columns"])
SENSOR_KEYS = tuple(f"s{index}" for index in range(len(SENSOR_COLUMNS)))
SENSOR_BY_KEY = dict(zip(SENSOR_KEYS, SENSOR_COLUMNS))
CSV_COLUMNS = ("Date",) + SENSOR_COLUMNS + ("Falha", "RUL")

EXPECTED_RAW_ROWS = int(DATASET_PROFILE["expected_raw_rows"])
EXPECTED_CLEAN_KEYS = int(DATASET_PROFILE["expected_clean_keys"])
EXPECTED_TARGET_CONFLICTS = dict(DATASET_PROFILE["expected_target_conflicts"])

# Proveniência explícita. O gerador herda o protocolo do notebook de origem;
# o gearbox é um CSV independente e não deriva dele.
GENERATOR_SOURCE_PROVENANCE = {
    "dataset_id": "gerador",
    "origin": "protocolo herdado de RUL_CMS_TCN.ipynb (datas FPT da célula 19)",
    "notebook": "RUL_CMS_TCN.ipynb",
    "sha256": "7a78c1b225cb8e2309585679e31927ae02e0287d3dfdc6d10f310cdc7409636d",
    "inherited_cell": 19,
    "replaced_draft": "RUL_CPFL_experimentos_colab.ipynb",
    "replaced_draft_sha256": "53e619b80c45815ef78f8d6997ea51d8bd21112cee15076643aa37c0c42b613f",
}
GEARBOX_SOURCE_PROVENANCE = {
    "dataset_id": "gearbox",
    "origin": "CSV de gearbox fornecido separadamente; não derivado de RUL_CMS_TCN.ipynb",
    "notebook": None,
    "sha256": None,
    "inherited_cell": None,
    "fpt_source": "perfil declarado 'gearbox': inferido da rotulagem linear, não verificado fisicamente",
}
SOURCE_PROVENANCE = (GENERATOR_SOURCE_PROVENANCE if DATASET_ID == "gerador"
                     else GEARBOX_SOURCE_PROVENANCE)

REPRESENTATIONS = ("current", "summary")
PREPROCESSING_OPTIONS = PREPROCESS_METHOD_CHOICES
WEIGHT_SCHEMES = ("none", "component", "component_phase")
N_ESTIMATORS = 128
FEATURE_CODE_VERSION = "causal-window-tabular-v2"
FORBIDDEN_INFERENCE_TOKENS = ("Falha", "Date", "RUL", "phase", "FPT",
                              "failure", "falha", "equipment")

MODEL_BOUNDS = {
    "ridge": {"alpha": (1e-6, 1e6)},
    "svr": {"C": (1e-6, 1e6), "epsilon": (0.0, 1e3)},
    "extra_trees": {"max_depth": (2, 16), "min_samples_leaf": (1, 50)},
}
SVR_EPSILON_UNITS_NOTE = (
    "epsilon do SVR é expresso na unidade do ALVO NORMALIZADO: RUL/100 para "
    "rótulos de dias e RUL (fração 0..1) para rótulos percentuais. Não é "
    "epsilon em dias nem em pontos percentuais."
)

print("Protocolo:", PROTOCOL["name"], PROTOCOL["revision"])
print("Dataset:", DATASET_ID, "| unit_kind:", UNIT_KIND, "| rótulos:", LABEL_ORDER)
print("Sensores:", len(SENSOR_COLUMNS), "| grupos:", EQUIPMENT_ORDER)
print("Pré-processamento:", PREPROCESSING_CONFIG)


In [ ]:
# =============================================================================
# SEÇÃO 3 — UTILITÁRIOS DE HASH, ESCRITA ATÔMICA E DESCOBERTA DE ENTRADA
# =============================================================================

import hashlib
import shutil
import stat as stat_module
import tempfile
import zipfile
from datetime import datetime as _datetime
from pathlib import PurePosixPath


def _json_default(value):
    if isinstance(value, (pd.Timestamp, _datetime)):
        return pd.Timestamp(value).isoformat()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, frozenset)):
        return sorted(value, key=lambda item: str(item))
    if isinstance(value, tuple):
        return list(value)
    raise TypeError(f"Objeto não serializável: {type(value)!r}")


def canonical_json(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
        allow_nan=False, default=_json_default,
    )


def sha256_json(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def sha256_text(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_bytes(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", dir=str(path.parent))
    try:
        with os.fdopen(fd, "wb") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def atomic_write_text(path, text):
    atomic_write_bytes(path, str(text).encode("utf-8"))


def atomic_write_json(path, payload):
    atomic_write_text(path, canonical_json(payload) + "\n")


def read_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def file_sha_map(paths):
    return {key: sha256_file(Path(path)) for key, path in sorted(paths.items())}


def _safe_zip_member(info):
    name = info.filename.replace("\\", "/")
    pure = PurePosixPath(name)
    if pure.is_absolute() or not pure.parts:
        return None
    if any(part in ("", ".", "..") for part in pure.parts):
        return None
    if ":" in pure.parts[0]:
        return None
    mode = (info.external_attr >> 16) & 0o170000
    if stat_module.S_ISLNK(mode):
        raise ValueError(f"ZIP contém link simbólico: {info.filename!r}")
    return pure


def safe_extract_csv_zip(zip_path, destination_root=INPUT_CACHE_DIR):
    """Valida e extrai somente CSVs de um ZIP, sem path traversal."""
    zip_path = Path(zip_path)
    if zip_path.suffix.lower() != ".zip" or not zip_path.is_file():
        raise ValueError(f"ZIP inválido: {zip_path}")
    digest = sha256_file(zip_path)
    destination_root = Path(destination_root)
    destination_root.mkdir(parents=True, exist_ok=True)
    destination = destination_root / f"zip-{digest[:16]}"
    marker = destination / ".complete"
    if marker.exists():
        return destination
    if destination.exists():
        raise RuntimeError(
            f"Extração incompleta existe em {destination}; remova-a explicitamente."
        )

    temporary = Path(tempfile.mkdtemp(prefix=f".zip-{digest[:8]}-",
                                      dir=str(destination_root)))
    extracted = 0
    total_uncompressed = 0
    try:
        with zipfile.ZipFile(zip_path, "r") as archive:
            infos = archive.infolist()
            if len(infos) > 2000:
                raise ValueError("ZIP com entradas demais.")
            for info in infos:
                total_uncompressed += int(info.file_size)
                if total_uncompressed > 2 * 1024 ** 3:
                    raise ValueError("ZIP excede o limite descompactado de 2 GiB.")
                if info.flag_bits & 0x1:
                    raise ValueError("ZIP criptografado não é aceito.")
                if info.file_size > 1024 ** 3:
                    raise ValueError(f"Entrada grande demais: {info.filename!r}")
                if (info.compress_size > 0 and info.file_size > 50 * 1024 ** 2
                        and info.file_size / max(info.compress_size, 1) > 200):
                    raise ValueError(f"Razão de compressão suspeita: {info.filename!r}")
                member = _safe_zip_member(info)
                if member is None:
                    raise ValueError(f"Caminho inseguro no ZIP: {info.filename!r}")
                if info.is_dir():
                    continue
                if member.suffix.lower() != ".csv":
                    continue
                target = (temporary / member).resolve()
                if temporary.resolve() not in target.parents:
                    raise ValueError(f"Destino fora da extração: {member}")
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info, "r") as source, open(target, "wb") as out:
                    shutil.copyfileobj(source, out, length=1024 * 1024)
                extracted += 1
        if extracted == 0:
            raise ValueError("ZIP não contém CSV.")
        atomic_write_text(temporary / ".complete",
                          canonical_json({"zip_sha256": digest,
                                          "csv_files": extracted}) + "\n")
        temporary.rename(destination)
        return destination
    except Exception:
        shutil.rmtree(temporary, ignore_errors=True)
        raise


def _upload_colab_files():
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError("Upload vazio ou cancelado.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    target_dir = INPUT_CACHE_DIR / "uploads" / stamp
    paths = []
    for original_name, payload in uploaded.items():
        safe_name = Path(original_name).name
        if not safe_name or safe_name != original_name:
            raise ValueError(f"Nome de upload inseguro: {original_name!r}")
        if Path(safe_name).suffix.lower() not in (".zip", ".csv"):
            raise ValueError(f"Upload deve ser ZIP ou CSV: {safe_name!r}")
        target = target_dir / safe_name
        atomic_write_bytes(target, payload)
        paths.append(target)
    return paths


def _label_for_selected_csv(file_name):
    """Rótulo do CSV selecionado; qualquer outro nome não pertence ao dataset."""
    matches = [label for label in LABEL_ORDER
               if PROTOCOL["labels"][label]["filename"] == file_name]
    if len(matches) > 1:
        raise ValueError(f"Mapeamento ambíguo para {file_name!r}.")
    return matches[0] if matches else None


def discover_input_files():
    """Descobre somente o CSV do dataset selecionado: sem merge e sem ambiguidade.

    Arquivos de outro dataset no mesmo diretório ou ZIP são ignorados. Um
    input.csv_paths de outro dataset gera erro; nada é selecionado por IDs.
    """
    input_cfg = CONFIG["input"]
    mode = input_cfg["mode"]
    selected_names = [PROTOCOL["labels"][label]["filename"] for label in LABEL_ORDER]
    roots = []
    explicit = {}
    origins = []

    if mode not in ("auto", "drive_dir", "files"):
        raise ValueError("input.mode deve ser auto, drive_dir ou files.")

    if mode in ("auto", "drive_dir"):
        drive_dir = Path(input_cfg["drive_dir"]) if input_cfg["drive_dir"] else None
        if drive_dir is not None and drive_dir.is_dir():
            roots.append(drive_dir)
            origins.append(f"diretorio:{drive_dir.name}")
        elif mode == "drive_dir":
            raise FileNotFoundError(f"input.drive_dir não existe: {drive_dir}")

    for zip_name in input_cfg["zip_paths"]:
        extracted = safe_extract_csv_zip(Path(zip_name))
        roots.append(extracted)
        origins.append(f"zip:{Path(zip_name).name}")

    for csv_name in input_cfg["csv_paths"]:
        path = Path(csv_name)
        if not path.is_file():
            raise FileNotFoundError(f"CSV explícito não existe: {path}")
        if path.suffix.lower() != ".csv":
            raise ValueError(f"input.csv_paths contém não-CSV: {path}")
        label = _label_for_selected_csv(path.name)
        if label is None:
            raise ValueError(
                f"CSV explícito {path.name!r} não é o arquivo do dataset selecionado "
                f"({DATASET_ID}: {', '.join(selected_names)}). Nenhum merge ou "
                "seleção por IDs sobrepostos é feito; ajuste CONFIG['dataset'] "
                "ou input.csv_paths."
            )
        if label in explicit and explicit[label].resolve() != path.resolve():
            raise ValueError(f"Dois CSVs explícitos para {label}.")
        explicit[label] = path
        origins.append(f"csv:{path.name}")

    found = {label: set() for label in LABEL_ORDER}
    for label, path in explicit.items():
        found[label].add(path.resolve())

    def scan_roots():
        for root in roots:
            for label in LABEL_ORDER:
                for path in Path(root).rglob(PROTOCOL["labels"][label]["filename"]):
                    if path.is_file():
                        found[label].add(path.resolve())

    scan_roots()

    missing = [label for label in LABEL_ORDER if not found[label]]
    if missing and mode == "auto" and input_cfg["allow_colab_upload"] and IN_COLAB:
        print("Entrada ausente:", ", ".join(missing))
        print("Solicitando upload do ZIP/CSV do dataset selecionado no Colab.")
        for uploaded_path in _upload_colab_files():
            if uploaded_path.suffix.lower() == ".zip":
                roots.append(safe_extract_csv_zip(uploaded_path))
                origins.append(f"upload_zip:{uploaded_path.name}")
            else:
                label = _label_for_selected_csv(uploaded_path.name)
                if label is None:
                    raise ValueError(
                        f"CSV enviado {uploaded_path.name!r} não é o arquivo do "
                        f"dataset selecionado ({DATASET_ID})."
                    )
                if label in explicit and explicit[label].resolve() != uploaded_path.resolve():
                    raise ValueError(f"Dois CSVs para {label} após upload.")
                explicit[label] = uploaded_path
                found[label].add(uploaded_path.resolve())
                origins.append(f"upload_csv:{uploaded_path.name}")
        scan_roots()

    missing = [label for label in LABEL_ORDER if not found[label]]
    if missing:
        raise FileNotFoundError(
            f"CSV do dataset {DATASET_ID} ausente: "
            + ", ".join(PROTOCOL["labels"][label]["filename"] for label in missing)
        )

    resolved = {}
    for label in LABEL_ORDER:
        candidates = sorted(found[label], key=lambda item: str(item))
        if len(candidates) != 1:
            raise RuntimeError(
                f"Descoberta ambígua para {label}: "
                + " | ".join(str(item) for item in candidates)
            )
        resolved[label] = candidates[0]
    return resolved, origins


print("Utilitários de I/O prontos.")


In [ ]:
# =============================================================================
# SEÇÃO 4 — LEITURA, VALIDAÇÃO, CONSOLIDAÇÃO E AUDITORIA
# A limpeza só agrupa dentro da mesma Falha e exige sensores idênticos.
# =============================================================================


coerce_fault_ids = _cer_rul.coerce_fault_ids


def load_raw_label(label, path):
    """Wrapper: ordem (label, path) -> núcleo (path, label) e constantes do notebook."""
    return _cer_rul.load_raw_label(
        path,
        label,
        expected_raw_rows=EXPECTED_RAW_ROWS,
        csv_columns=CSV_COLUMNS,
        sensor_columns=SENSOR_COLUMNS,
        fault_ids=FAULT_IDS,
        max_allowed=PROTOCOL["labels"][label]["max_allowed"],
        check_positive_sensors=True,
    )


def consolidate_frame(frame, label, enforce_expectations=True):
    """Wrapper: constantes do notebook e semântica original de enforce/fallback."""
    label_meta = PROTOCOL["labels"].get(label, {
        "divisor": 1.0, "metric_factor": 1.0, "unit": "unidade",
        "kind": "synthetic", "max_allowed": np.inf,
    })
    return _cer_rul.consolidate_frame(
        frame,
        label,
        sensor_columns=SENSOR_COLUMNS,
        csv_columns=CSV_COLUMNS,
        fault_ids=FAULT_IDS,
        id_to_equipment=ID_TO_EQUIPMENT,
        equipment_order=EQUIPMENT_ORDER,
        label_meta=label_meta,
        expected_clean_keys=EXPECTED_CLEAN_KEYS if enforce_expectations else None,
        expected_target_conflicts=(
            EXPECTED_TARGET_CONFLICTS[label] if enforce_expectations else None
        ),
        enforce_expectations=enforce_expectations,
    )


def clean_frame_fingerprint(clean):
    rows = []
    for record in clean.to_dict(orient="records"):
        rows.append([
            int(record["Falha"]),
            pd.Timestamp(record["Date"]).isoformat(),
            *[float(record[column]).hex() for column in SENSOR_COLUMNS],
            float(record["RUL"]).hex(),
            int(record["raw_count"]),
        ])
    return sha256_json(rows)


def key_fingerprint(clean):
    return sha256_json([
        [int(record["Falha"]), pd.Timestamp(record["Date"]).isoformat()]
        for record in clean.to_dict(orient="records")
    ])


def sensor_fingerprint(clean):
    return sha256_json([
        [int(record["Falha"]), pd.Timestamp(record["Date"]).isoformat(),
         *[float(record[column]).hex() for column in SENSOR_COLUMNS]]
        for record in clean.to_dict(orient="records")
    ])


def assert_raw_alignment(frames):
    reference_label = LABEL_ORDER[0]
    reference = frames[reference_label]
    for label in LABEL_ORDER[1:]:
        frame = frames[label]
        if len(frame) != len(reference):
            raise ValueError(f"Alinhamento bruto: {label} tem linhas diferentes.")
        if not frame.loc[:, ["Falha", "Date"]].equals(reference.loc[:, ["Falha", "Date"]]):
            raise ValueError(f"Alinhamento bruto: chaves/ordem divergem em {label}.")
        if not np.array_equal(
            frame.loc[:, SENSOR_COLUMNS].to_numpy(float),
            reference.loc[:, SENSOR_COLUMNS].to_numpy(float),
        ):
            raise ValueError(f"Alinhamento bruto: sensores divergem em {label}.")
    return True


def assert_clean_alignment(cleaned):
    reference_label = LABEL_ORDER[0]
    reference = cleaned[reference_label]
    for label in LABEL_ORDER[1:]:
        frame = cleaned[label]
        if len(frame) != len(reference):
            raise ValueError(f"Alinhamento limpo: {label} tem chaves diferentes.")
        if not frame.loc[:, ["Falha", "Date"]].equals(reference.loc[:, ["Falha", "Date"]]):
            raise ValueError(f"Alinhamento limpo: chaves/ordem divergem em {label}.")
        if not np.array_equal(
            frame.loc[:, SENSOR_COLUMNS].to_numpy(float),
            reference.loc[:, SENSOR_COLUMNS].to_numpy(float),
        ):
            raise ValueError(f"Alinhamento limpo: sensores divergem em {label}.")
    return True


def load_and_consolidate_all():
    files, origins = discover_input_files()
    raw = {label: load_raw_label(label, files[label]) for label in LABEL_ORDER}
    assert_raw_alignment(raw)
    cleaned = {}
    audits = {}
    terminals = {}
    for label in LABEL_ORDER:
        clean, audit, terminal = consolidate_frame(raw[label], label,
                                                   enforce_expectations=True)
        cleaned[label] = clean
        audits[label] = audit
        terminals[label] = terminal
    assert_clean_alignment(cleaned)
    hashes = file_sha_map(files)
    for label in LABEL_ORDER:
        print(
            f"{label}: raw={audits[label]['raw_rows']} | "
            f"chaves={audits[label]['raw_unique_keys']} | "
            f"conflitos_alvo={audits[label]['target_conflict_keys']} | "
            f"terminais!=0={audits[label]['terminal_nonzero_count']}"
        )
    return files, origins, raw, cleaned, audits, terminals, hashes


print("Funções de dados prontas.")


In [ ]:
# =============================================================================
# SEÇÃO 4 (execução) — CARGA E AUDITORIA DO DATASET SELECIONADO
# =============================================================================

INPUT_FILES, INPUT_ORIGINS, RAW, CLEAN, AUDITS, TERMINALS, RAW_FILE_HASHES = (
    load_and_consolidate_all()
)

CLEAN_HASHES = {label: clean_frame_fingerprint(CLEAN[label]) for label in LABEL_ORDER}
KEY_HASH = key_fingerprint(CLEAN[LABEL_ORDER[0]])
SENSOR_HASH = sensor_fingerprint(CLEAN[LABEL_ORDER[0]])

DATA_BASE_IDENTITY = {
    "dataset_id": DATASET_ID,
    "unit_kind": UNIT_KIND,
    "csv_columns": list(CSV_COLUMNS),
    "label_filenames": {label: PROTOCOL["labels"][label]["filename"]
                        for label in LABEL_ORDER},
    "source_notebook_sha256": SOURCE_PROVENANCE["sha256"],
    "protocol_revision": PROTOCOL["revision"],
    "raw_file_sha256": RAW_FILE_HASHES,
    "clean_frame_sha256": CLEAN_HASHES,
    "key_sha256": KEY_HASH,
    "sensor_matrix_sha256": SENSOR_HASH,
}
DATA_BASE_HASH = sha256_json(DATA_BASE_IDENTITY)

AUDIT_ROWS = []
for label in LABEL_ORDER:
    audit = AUDITS[label]
    AUDIT_ROWS.append({
        "label": label,
        "raw_rows": audit["raw_rows"],
        "raw_unique_keys": audit["raw_unique_keys"],
        "raw_duplicate_keys": audit["raw_duplicate_keys"],
        "raw_max_multiplicity": audit["raw_max_multiplicity"],
        "exact_duplicate_groups": audit["exact_duplicate_groups"],
        "exact_duplicate_surplus_rows": audit["exact_duplicate_surplus_rows"],
        "target_conflict_keys": audit["target_conflict_keys"],
        "feature_conflict_keys": audit["feature_conflict_keys"],
        "clean_keys": audit["clean_keys"],
        "terminal_nonzero_count": audit["terminal_nonzero_count"],
        "raw_RUL_min": audit["raw_target_stats"]["min"],
        "raw_RUL_max": audit["raw_target_stats"]["max"],
        "clean_RUL_min": audit["clean_target_stats"]["min"],
        "clean_RUL_max": audit["clean_target_stats"]["max"],
    })
AUDIT_TABLE = pd.DataFrame(AUDIT_ROWS)
TERMINAL_TABLE = pd.concat(
    [TERMINALS[label].assign(label=label) for label in LABEL_ORDER],
    ignore_index=True,
)
TERMINAL_TABLE = TERMINAL_TABLE.loc[:, ["label", "Falha", "Date", "RUL"]]

print("Arquivos de entrada:")
for label in LABEL_ORDER:
    print(f"  {label}: {INPUT_FILES[label].name} | sha256={RAW_FILE_HASHES[label][:12]}…")
print("Origens:", ", ".join(INPUT_ORIGINS))
print("\nAuditoria bruta/limpa:")
display(AUDIT_TABLE)
print("\nAlvos terminais (nenhum zero foi imposto manualmente):")
display(TERMINAL_TABLE)
print("Chaves limpas totais por rótulo:", {label: len(CLEAN[label]) for label in LABEL_ORDER})
print("Hash base dos dados:", DATA_BASE_HASH)


In [ ]:
# =============================================================================
# CÉLULA EDA COMPARTILHÁVEL — SENSORES LIMPOS E CADÊNCIA POR FALHA
# Leve: não ajusta modelos e não depende de nenhum resultado de treino.
# =============================================================================


def build_sensor_quantiles(clean_reference):
    rows = []
    for column in SENSOR_COLUMNS:
        values = clean_reference[column].astype(float)
        rows.append({
            "sensor": column,
            "n": int(values.size),
            "min": float(values.min()),
            "q01": float(values.quantile(0.01)),
            "q05": float(values.quantile(0.05)),
            "q25": float(values.quantile(0.25)),
            "median": float(values.quantile(0.50)),
            "q75": float(values.quantile(0.75)),
            "q95": float(values.quantile(0.95)),
            "q99": float(values.quantile(0.99)),
            "max": float(values.max()),
            "mean": float(values.mean()),
            "std": float(values.std(ddof=0)),
        })
    return pd.DataFrame(rows)


def build_time_gap_table(clean_reference):
    rows = []
    for fault_id, group in clean_reference.groupby("Falha", sort=True):
        group = group.sort_values("Date", kind="mergesort")
        dates = group["Date"]
        gaps_h = dates.diff().dt.total_seconds().dropna() / 3600.0
        rows.append({
            "Falha": int(fault_id),
            "equipment": ID_TO_EQUIPMENT[int(fault_id)],
            "n_records": int(len(group)),
            "first_date": dates.iloc[0].isoformat(),
            "last_date": dates.iloc[-1].isoformat(),
            "duration_days": float(
                (dates.iloc[-1] - dates.iloc[0]).total_seconds() / 86400.0
            ),
            "median_gap_h": float(gaps_h.median()) if len(gaps_h) else None,
            "mean_gap_h": float(gaps_h.mean()) if len(gaps_h) else None,
            "min_gap_h": float(gaps_h.min()) if len(gaps_h) else None,
            "max_gap_h": float(gaps_h.max()) if len(gaps_h) else None,
            "q95_gap_h": float(gaps_h.quantile(0.95)) if len(gaps_h) else None,
        })
    return pd.DataFrame(rows)


# Os oito sensores e as chaves temporais são idênticos entre rótulos; a
# referência evita repetir exatamente a mesma EDA por rótulo.
SENSOR_EDA = build_sensor_quantiles(CLEAN[LABEL_ORDER[0]])
TIME_GAP_EDA = build_time_gap_table(CLEAN[LABEL_ORDER[0]])

print("CÉLULA EDA COMPARTILHÁVEL — quantis dos oito sensores limpos:")
display(SENSOR_EDA)
print("Intervalos temporais por Falha (horas; datas reais):")
display(TIME_GAP_EDA)


In [ ]:
# =============================================================================
# SEÇÃO 5 — SPLITS EXPLÍCITOS: LEAVE-ONE-GROUP-OUT (um fold por grupo)
# =============================================================================

def make_splits():
    return _cer_rul.make_splits(
        equipment_order=EQUIPMENT_ORDER,
        equipment_groups=PROTOCOL["equipment_groups"],
    )


SPLITS = make_splits()
SPLIT_HASH = sha256_json(SPLITS.to_dict(orient="records"))

# Invariantes do protocolo.
assert len(SPLITS) == len(EQUIPMENT_ORDER)
assert set(SPLITS["test_equipment"]) == set(EQUIPMENT_ORDER)
assert len(set(SPLITS["val_equipment"])) >= 1
for row in SPLITS.to_dict(orient="records"):
    test = set(row["test_ids"])
    val = set(row["val_ids"])
    inner = set(row["inner_train_ids"])
    outer = set(row["outer_train_ids"])
    assert test.isdisjoint(val | inner)
    assert val.isdisjoint(inner)
    assert val | inner == outer
    assert outer.isdisjoint(test)
    assert test | outer == set(FAULT_IDS)
    assert len(test) > 0 and len(val) > 0 and len(inner) > 0
    assert len(row["inner_train_equipment"]) == len(EQUIPMENT_ORDER) - 2

print("Splits (um grupo de teste e um de validação por fold; unidade:",
      UNIT_KIND, "):")
display(SPLITS.loc[:, ["fold_id", "test_ids", "val_ids", "inner_train_ids",
                       "outer_train_ids"]])
print("Hash dos splits:", SPLIT_HASH)


## Endpoints: ciclo pós-aquecimento, não o dataset inteiro

A lista de endpoints é construída **uma vez** a partir das chaves limpas e é reutilizada por todos os rótulos, larguras de grade, candidatos, modelos e níveis de ablação.

- Ordenação cronológica dentro de cada `Falha` (nunca entre Falhas).
- Aquecimento: os primeiros **30 registros únicos** não geram endpoint.
- Depois do aquecimento: índices `30, 33, 36, ...`, com passo 3.
- O **último registro** de cada Falha é sempre incluído, mesmo que o passo não caia nele.
- O hash da lista e um CSV com `endpoint_id`, Falha, equipamento, data, índice e contagem de contexto são exportados.

Isso significa que os resultados descrevem o ciclo **posterior ao aquecimento**. Nenhuma métrica deve ser apresentada como cobertura de todas as linhas originais.


In [ ]:
# =============================================================================
# SEÇÃO 6 — LISTA COMPARTILHADA DE ENDPOINTS
# =============================================================================

EXPECTED_ENDPOINT_COUNT = int(DATASET_PROFILE["expected_endpoints"])


def phase_for_date(date_value, fault_id):
    return _cer_rul.phase_for_date(
        date_value, fault_id,
        fpt_annotations=PROTOCOL["fpt_annotations"],
    )


def build_endpoint_rows(clean_reference, warmup=None, stride=None, enforce=True):
    return _cer_rul.build_endpoint_rows(
        clean_reference,
        warmup=(PROTOCOL["warmup_records"] if warmup is None else int(warmup)),
        stride=(PROTOCOL["endpoint_stride"] if stride is None else int(stride)),
        id_to_equipment=ID_TO_EQUIPMENT,
        fpt_annotations=PROTOCOL["fpt_annotations"],
        expected_endpoint_count=EXPECTED_ENDPOINT_COUNT,
        enforce=enforce,
    )


ENDPOINTS = build_endpoint_rows(CLEAN[LABEL_ORDER[0]])
ENDPOINT_HASH = sha256_json([
    [row.endpoint_id, int(row.Falha), pd.Timestamp(row.Date).isoformat(),
     int(row.record_index)]
    for row in ENDPOINTS.itertuples(index=False)
])
ENDPOINT_COUNTS = {
    str(int(fid)): int(count)
    for fid, count in ENDPOINTS.groupby("Falha").size().items()
}

TARGETS = {}
for label in LABEL_ORDER:
    target = ENDPOINTS.merge(
        CLEAN[label].loc[:, ["Falha", "Date", "RUL"]],
        on=["Falha", "Date"], how="left", sort=False, validate="one_to_one",
    )
    if target["RUL"].isna().any():
        raise ValueError(f"{label}: alvo ausente em algum endpoint.")
    if target["endpoint_id"].tolist() != ENDPOINTS["endpoint_id"].tolist():
        raise ValueError(f"{label}: ordem dos endpoints mudou no merge.")
    meta = PROTOCOL["labels"][label]
    target["label"] = label
    target["y_scaled"] = target["RUL"].astype(float) / float(meta["divisor"])
    target["y_metric"] = target["RUL"].astype(float) * float(meta["metric_factor"])
    target["y_unit"] = meta["unit"]
    TARGETS[label] = target

assert ENDPOINTS["Falha"].map(ID_TO_EQUIPMENT).equals(ENDPOINTS["equipment"])
assert all(len(TARGETS[label]) == len(ENDPOINTS) for label in LABEL_ORDER)
assert all(TARGETS[label]["endpoint_id"].tolist() == ENDPOINTS["endpoint_id"].tolist()
           for label in LABEL_ORDER)

print("Endpoints totais:", len(ENDPOINTS), "| por Falha:", ENDPOINT_COUNTS)
print("Aquecimento:", PROTOCOL["warmup_records"], "registros; passo:",
      PROTOCOL["endpoint_stride"], "; final sempre incluído.")
print("Hash dos endpoints:", ENDPOINT_HASH)
display(ENDPOINTS.groupby(["Falha", "equipment"], sort=True)
        .agg(endpoints=("endpoint_id", "size"),
             primeiro=("Date", "min"), ultimo=("Date", "max"))
        .reset_index())


## Features tabulares: somente a janela de 32 bins

A matriz tabular é deliberadamente compacta e **não recebe os 32 valores brutos por bin**:

- `current`: valor observado mais recente de cada sensor **dentro da janela de 32 bins**, contagem de registros por sensor, fração de bins vazios, número de registros na janela, bins observados e bins vazios.
- `summary`: `current` + média, desvio e inclinação por **dia real** calculados sobre os valores finais dos bins observados, usando seus tempos reais.
- `timing` opcional: duração fixa da janela (`32 × largura`), duração observada entre o primeiro e o último bin ocupado, gap do registro imediatamente anterior a `t` (que pode estar fora da grade) e mediana dos intervalos entre registros dentro da grade.

Toda estatística de valor usa somente os bins da janela `(t − 32·w, t]`. Registros antigos fora da grade não alteram `current` nem `summary`, embora possam aparecer apenas no gap anterior de timing. O contexto bruto dos bins não é adicionado à matriz; se um modelo Timestral precisar dele, deverá ser cacheado separadamente em uma tarefa própria.


In [ ]:
# =============================================================================
# SEÇÃO 7 — FEATURES CAUSAIS TABULARES
# A matriz tabular usa SOMENTE a janela regular de 32 bins ancorada à direita em t.
# Valores brutos por bin NÃO entram na matriz. O contexto bruto para um eventual
# modelo Timestral futuro deve ser cacheado separadamente, em outra tarefa.
# =============================================================================


def feature_columns(width_hours, representation, timing):
    return _cer_rul.feature_columns(
        width_hours, representation, timing,
        sensor_keys=SENSOR_KEYS,
        representations=REPRESENTATIONS,
        bin_hours_candidates=PROTOCOL["bin_hours_candidates"],
        forbidden_tokens=FORBIDDEN_INFERENCE_TOKENS,
    )


_daily_stats = _cer_rul.daily_stats


def _bin_stats(context_dates, context_values, endpoint_date, width_hours):
    return _cer_rul.bin_stats(
        context_dates, context_values, endpoint_date, width_hours,
        context_bins=PROTOCOL["context_bins"],
    )


def build_feature_row(endpoint_row, clean_group, width_hours):
    return _cer_rul.build_feature_row(
        endpoint_row, clean_group, width_hours,
        sensor_columns=SENSOR_COLUMNS,
        sensor_keys=SENSOR_KEYS,
        context_bins=PROTOCOL["context_bins"],
    )


def build_feature_frame(clean_reference, endpoints, width_hours):
    return _cer_rul.build_feature_frame(
        clean_reference, endpoints, width_hours,
        sensor_columns=SENSOR_COLUMNS,
        sensor_keys=SENSOR_KEYS,
        context_bins=PROTOCOL["context_bins"],
        bin_hours_candidates=PROTOCOL["bin_hours_candidates"],
        representations=REPRESENTATIONS,
        forbidden_tokens=FORBIDDEN_INFERENCE_TOKENS,
    )


def feature_cache_key(width_hours):
    return sha256_json({
        "feature_code_version": FEATURE_CODE_VERSION,
        "protocol_revision": PROTOCOL["revision"],
        "dataset_id": DATASET_ID,
        "data_base_hash": DATA_BASE_HASH,
        "endpoint_hash": ENDPOINT_HASH,
        "width_hours": int(width_hours),
        "n_bins": PROTOCOL["context_bins"],
        "sensor_keys": list(SENSOR_KEYS),
    })


FEATURE_MATRICES = {}


def get_feature_matrix(width_hours):
    width_hours = int(width_hours)
    if width_hours in FEATURE_MATRICES:
        return FEATURE_MATRICES[width_hours]
    key = feature_cache_key(width_hours)
    path = FEATURE_CACHE_DIR / f"{key}.npz"
    expected_columns = feature_columns(width_hours, "summary", True)
    if path.exists():
        with np.load(path, allow_pickle=False) as cached:
            stored_key = str(cached["cache_key"].item())
            stored_columns = json.loads(str(cached["columns"].item()))
            stored_ids = [str(value) for value in cached["endpoint_ids"].tolist()]
            if stored_key != key:
                raise RuntimeError("Cache de features com identidade incompatível.")
            if stored_columns != expected_columns:
                raise RuntimeError("Cache de features com colunas incompatíveis.")
            if stored_ids != ENDPOINTS["endpoint_id"].tolist():
                raise RuntimeError("Cache de features com endpoints incompatíveis.")
            matrix = pd.DataFrame(
                np.asarray(cached["X"], dtype=float),
                index=pd.Index(stored_ids, name="endpoint_id"),
                columns=stored_columns,
            )
        print(f"Features {width_hours} h: cache válido ({matrix.shape}).")
    else:
        matrix = build_feature_frame(CLEAN[LABEL_ORDER[0]], ENDPOINTS, width_hours)
        temporary = path.with_name(f".{path.stem}.tmp.npz")
        np.savez_compressed(
            temporary,
            X=matrix.to_numpy(float),
            columns=np.array(json.dumps(matrix.columns.tolist())),
            endpoint_ids=np.array(matrix.index.tolist()),
            cache_key=np.array(key),
        )
        os.replace(temporary, path)
        print(f"Features {width_hours} h: construídas ({matrix.shape}) e cacheadas.")
    if matrix.isna().all(axis=0).any():
        print(f"  colunas totalmente vazias em {width_hours} h:",
              int(matrix.isna().all(axis=0).sum()))
    FEATURE_MATRICES[width_hours] = matrix
    return matrix


print("Funções de features prontas. Dimensão summary/current por largura:")
for _width in PROTOCOL["bin_hours_candidates"]:
    print(f"  {_width} h: summary={len(feature_columns(_width, 'summary', True))} | "
          f"current={len(feature_columns(_width, 'current', True))}")


## Atributos de forecast

O Timestral prevê somente os oito sensores. O contexto tem forma `(B, 8, 32)` e vem da mesma grade de 32 bins `(left, right]` do protocolo compartilhado, usando o último valor observado por sensor e bin; bins vazios são NaN. Nenhum valor posterior ao endpoint entra no contexto.

Para cada sensor e horizonte, cinco atributos resumem a trajetória prevista:

- `fc_lastmedian__s`: mediana no último passo do horizonte;
- `fc_meanmedian__s`: média das medianas no horizonte;
- `fc_delta__s`: última mediana menos o último valor observado;
- `fc_slope_day__s`: inclinação da mediana por dia real de forecast;
- `fc_width__s`: média de `q90 - q10` no horizonte.

Eles são concatenados aos atributos observados `current`/`summary`. As trajetórias futuras brutas não são adicionadas à matriz tabular. A largura `q90 - q10` é uma dispersão preditiva dos sensores, não um intervalo de confiança da RUL.

O cache contém somente medianas, quantis, atributos e metadados sensor-only. Sua chave inclui hash dos sensores, endpoints, grade, comprimento do contexto, horizonte, repositório/pesos, commit do pacote, opções, device e dtype; **não inclui rótulo, alvo ou hash de alvo**. O mesmo forecast congelado é reutilizado pelos cinco rótulos, folds e tentativas.

O preview de custo (`unique_configs`, batches e endpoints faltantes) é impresso antes de qualquer carregamento de pesos ou inferência. O contador de inferência é exportado separadamente do contador de ajustes supervisionados.


In [ ]:
# =============================================================================
# SEÇÃO 10 — CONTEXTO, FORECAST CONGELADO E ATRIBUTOS DE FORECAST
# A RUL nunca entra no contexto nem na identidade do cache.
# =============================================================================

TIMESTRAL = CONFIG["model_specific"]
TIMESTRAL_CACHE_DIR = CACHE_DIR / TIMESTRAL["cache_subdir"]
TIMESTRAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TIMESTRAL_FEATURE_MATRICES = {}
TIMESTRAL_CACHE_METADATA = {}
FORECAST_COUNTER = {}


def horizon_steps(horizon_days, bin_hours):
    days = float(horizon_days)
    width = int(bin_hours)
    if width not in PROTOCOL["bin_hours_candidates"]:
        raise ValueError(f"Largura de grade não aprovada: {bin_hours}")
    if not (days > 0):
        raise ValueError("horizon_days deve ser positivo.")
    hours = days * 24.0
    steps = hours / width
    if not np.isfinite(steps) or abs(steps - round(steps)) > 1e-9:
        raise ValueError(
            f"Horizonte {horizon_days} d não é múltiplo inteiro de {bin_hours} h."
        )
    steps = int(round(steps))
    if steps < 1:
        raise ValueError("Horizonte produz menos de um passo.")
    return steps


def timestral_forecast_columns():
    columns = []
    for sensor_key in SENSOR_KEYS:
        columns.extend([
            f"fc_lastmedian__{sensor_key}",
            f"fc_meanmedian__{sensor_key}",
            f"fc_delta__{sensor_key}",
            f"fc_slope_day__{sensor_key}",
            f"fc_width__{sensor_key}",
        ])
    if len(set(columns)) != len(columns):
        raise ValueError("Colunas de forecast duplicadas.")
    for column in columns:
        if any(token in column for token in FORBIDDEN_INFERENCE_TOKENS):
            raise ValueError(f"Feature de forecast proibida: {column}")
    return tuple(columns)


TIMESTRAL_FORECAST_COLUMNS = timestral_forecast_columns()


def timestral_feature_columns(width_hours, representation, timing):
    return tuple(feature_columns(width_hours, representation, timing)) + (
        TIMESTRAL_FORECAST_COLUMNS
    )


def build_timestral_contexts(clean_reference, endpoints, width_hours):
    """(N, 8, 32) com os últimos valores observados por bin, NaN quando vazio."""
    width_hours = int(width_hours)
    if width_hours not in PROTOCOL["bin_hours_candidates"]:
        raise ValueError(f"Largura não aprovada: {width_hours}.")
    if int(TIMESTRAL["context_length"]) != int(PROTOCOL["context_bins"]):
        raise ValueError("context_length do Timestral deve coincidir com os 32 bins.")
    if len(endpoints) == 0:
        return np.empty((0, len(SENSOR_KEYS), PROTOCOL["context_bins"]), dtype=float)
    groups = {
        int(fault_id): group.sort_values("Date", kind="mergesort").reset_index(drop=True)
        for fault_id, group in clean_reference.groupby("Falha", sort=True)
    }
    contexts = np.full(
        (len(endpoints), len(SENSOR_KEYS), PROTOCOL["context_bins"]),
        np.nan, dtype=float,
    )
    for position, endpoint_row in enumerate(endpoints.reset_index(drop=True).itertuples(index=False)):
        group = groups[int(endpoint_row.Falha)]
        record_index = int(endpoint_row.record_index)
        context = group.iloc[:record_index + 1]
        endpoint_date = pd.Timestamp(endpoint_row.Date)
        if len(context) != record_index + 1:
            raise ValueError("Contexto truncado.")
        if pd.Timestamp(context["Date"].iloc[-1]) != endpoint_date:
            raise ValueError("O endpoint não é o último registro do contexto.")
        if (context["Date"] > endpoint_date).any():
            raise ValueError("Contexto contém valor futuro.")
        last_values, _, counts = _bin_stats(
            context["Date"], context.loc[:, SENSOR_COLUMNS].to_numpy(float),
            endpoint_date, width_hours,
        )
        if last_values.shape != (PROTOCOL["context_bins"], len(SENSOR_KEYS)):
            raise ValueError("Forma inesperada de _bin_stats.")
        contexts[position] = last_values.T
    if not np.isfinite(contexts).any():
        raise ValueError("Nenhum contexto finito.")
    return contexts


def current_observed_matrix(width_hours, endpoint_ids):
    observed = get_feature_matrix(width_hours)
    columns = [f"cur__{sensor_key}" for sensor_key in SENSOR_KEYS]
    current = observed.loc[endpoint_ids, columns].to_numpy(float)
    if current.shape != (len(endpoint_ids), len(SENSOR_KEYS)):
        raise ValueError("Matriz current com forma inesperada.")
    return current


def validate_forecast_arrays(median, q10, q90, n_endpoints, steps):
    arrays = []
    for name, array in (("median", median), ("q10", q10), ("q90", q90)):
        array = np.asarray(array, dtype=float)
        expected = (int(n_endpoints), len(SENSOR_KEYS), int(steps))
        if array.shape != expected:
            raise ValueError(f"{name}: forma {array.shape}, esperado {expected}.")
        if not np.isfinite(array).all():
            raise ValueError(f"{name}: forecast contém NaN/infinito.")
        arrays.append(array)
    median, q10, q90 = arrays
    tolerance = 1e-6 * max(1.0, float(np.max(np.abs(median))))
    if not np.all(q10 <= median + tolerance):
        raise ValueError("Forecast viola q10 <= mediana.")
    if not np.all(median <= q90 + tolerance):
        raise ValueError("Forecast viola mediana <= q90.")
    return tuple(arrays)


def validate_forecast_feature_frame(frame):
    if not isinstance(frame, pd.DataFrame):
        raise ValueError("Features de forecast devem ser DataFrame.")
    if frame.columns.tolist() != list(TIMESTRAL_FORECAST_COLUMNS):
        raise ValueError("Ordem/colunas das features de forecast mudaram.")
    values = frame.to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError(
            "Features de forecast contêm NaN/infinito; todos os 40 atributos "
            "devem ser finitos no endpoint observado."
        )
    return frame


def forecast_cache_metadata(width_hours, horizon_days, cache_key, endpoint_ids,
                            actual_device):
    return {
        "identity": timestral_cache_identity_payload(width_hours, horizon_days),
        "cache_key": str(cache_key),
        "endpoint_ids": [str(value) for value in endpoint_ids],
        "actual_device": str(actual_device),
        "forecast_feature_columns": list(TIMESTRAL_FORECAST_COLUMNS),
    }


def assert_forecast_cache_metadata(metadata, width_hours, horizon_days,
                                   cache_key, endpoint_ids):
    expected_identity = timestral_cache_identity_payload(width_hours, horizon_days)
    if metadata.get("cache_key") != str(cache_key):
        raise RuntimeError("Cache de forecast com cache_key incompatível.")
    if metadata.get("identity") != expected_identity:
        raise RuntimeError("Cache de forecast com identidade completa incompatível.")
    if metadata.get("endpoint_ids") != [str(value) for value in endpoint_ids]:
        raise RuntimeError("Cache de forecast com endpoints incompatíveis.")
    if metadata.get("forecast_feature_columns") != list(TIMESTRAL_FORECAST_COLUMNS):
        raise RuntimeError("Cache de forecast com colunas incompatíveis.")
    if not isinstance(metadata.get("actual_device"), str):
        raise RuntimeError("Cache de forecast sem device real registrado.")
    return metadata


def forecast_feature_frame(endpoint_ids, median, q10, q90, current,
                           width_hours, horizon_days):
    steps = horizon_steps(horizon_days, width_hours)
    median, q10, q90 = validate_forecast_arrays(
        median, q10, q90, len(endpoint_ids), steps)
    current = np.asarray(current, dtype=float)
    if current.shape != (len(endpoint_ids), len(SENSOR_KEYS)):
        raise ValueError("Current e endpoints desalinhados.")
    days = np.arange(1, steps + 1, dtype=float) * float(width_hours) / 24.0
    centered = days - days.mean()
    denominator = float(np.dot(centered, centered))
    columns = {}
    for sensor_position, sensor_key in enumerate(SENSOR_KEYS):
        med = median[:, sensor_position, :]
        last_median = med[:, -1]
        mean_median = med.mean(axis=1)
        delta = last_median - current[:, sensor_position]
        if steps > 1 and denominator > 0:
            slope = ((med - mean_median[:, None]) * centered[None, :]).sum(axis=1) / denominator
        else:
            slope = np.zeros(len(endpoint_ids), dtype=float)
        width_mean = (q90[:, sensor_position, :]
                      - q10[:, sensor_position, :]).mean(axis=1)
        columns[f"fc_lastmedian__{sensor_key}"] = last_median
        columns[f"fc_meanmedian__{sensor_key}"] = mean_median
        columns[f"fc_delta__{sensor_key}"] = delta
        columns[f"fc_slope_day__{sensor_key}"] = slope
        columns[f"fc_width__{sensor_key}"] = width_mean
    frame = pd.DataFrame(columns, index=pd.Index(endpoint_ids, name="endpoint_id"))
    return validate_forecast_feature_frame(frame)


def timestral_cache_identity_payload(width_hours, horizon_days):
    """Identidade sensor-only; nenhum rótulo, alvo ou hash de alvo participa."""
    steps = horizon_steps(horizon_days, width_hours)
    return {
        "protocol_revision": PROTOCOL["revision"],
        "dataset_id": DATASET_ID,
        "sensor_sha256": SENSOR_HASH,
        "endpoint_sha256": ENDPOINT_HASH,
        "context_bins": int(PROTOCOL["context_bins"]),
        "context_length": int(TIMESTRAL["context_length"]),
        "width_hours": int(width_hours),
        "horizon_days": float(horizon_days),
        "horizon_steps": int(steps),
        "model_repo": TIMESTRAL["model_repo"],
        "weights_revision": TIMESTRAL["model_revision"],
        "package_commit": TIMESTRAL["package_commit"],
        "symmetric_averaging": bool(TIMESTRAL["symmetric_averaging"]),
        "clip_nonnegative": bool(TIMESTRAL["clip_nonnegative"]),
        "device": str(TIMESTRAL["device"]),
        "dtype": "float32",
        "forecast_feature_version": TIMESTRAL["forecast_feature_version"],
    }


def timestral_cache_key(width_hours, horizon_days):
    return sha256_json(timestral_cache_identity_payload(width_hours, horizon_days))


def timestral_cache_path(width_hours, horizon_days):
    return TIMESTRAL_CACHE_DIR / f"{timestral_cache_key(width_hours, horizon_days)}.npz"


def _forecast_counter_add(batches=0, endpoints=0):
    FORECAST_COUNTER["batches"] = FORECAST_COUNTER.get("batches", 0) + int(batches)
    FORECAST_COUNTER["endpoints"] = FORECAST_COUNTER.get("endpoints", 0) + int(endpoints)


def _resolve_device():
    requested = str(TIMESTRAL["device"]).lower()
    if requested not in ("cpu", "cuda", "auto"):
        raise ValueError("model_specific.device deve ser cpu, cuda ou auto.")
    import torch
    if requested == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("device='cuda' solicitado, mas CUDA não está disponível.")
    return requested


def load_timestral_forecaster():
    source_audit = validate_timestral_runtime()
    try:
        import torch
        from timestral import TimestralForecaster
    except Exception as exc:
        raise RuntimeError(
            "Timestral/torch não instalados. No Colab, ative "
            "model_specific.install_dependencies=True (URL fixada) ou instale "
            "o pacote aprovado; o smoke de dados não exige Timestral."
        ) from exc
    device = _resolve_device()
    forecaster = TimestralForecaster.from_pretrained(
        TIMESTRAL["model_repo"],
        revision=TIMESTRAL["model_revision"],
        device=device,
    )
    actual_device = str(forecaster.device)
    TIMESTRAL_SOURCE_AUDIT.clear()
    TIMESTRAL_SOURCE_AUDIT.update(source_audit)
    TIMESTRAL_VERSIONS.update({
        **source_audit,
        "torch": getattr(torch, "__version__", _metadata_version("torch")),
        "timestral": _metadata_version("timestral"),
        "actual_device": actual_device,
        "configured_package_commit": TIMESTRAL["package_commit"],
        "verified_package_commit": source_audit["verified_package_commit"],
        "weights_revision": TIMESTRAL["model_revision"],
    })
    print("Timestral carregado:", TIMESTRAL["model_repo"],
          "| revision:", TIMESTRAL["model_revision"],
          "| device:", actual_device)
    return forecaster


def _forecast_batch(forecaster, contexts, steps):
    output = forecaster.forecast(
        contexts,
        prediction_length=int(steps),
        symmetric_averaging=bool(TIMESTRAL["symmetric_averaging"]),
        clip_nonnegative=bool(TIMESTRAL["clip_nonnegative"]),
    )
    median = output.quantile(0.5).detach().cpu().numpy()
    q10 = output.quantile(0.1).detach().cpu().numpy()
    q90 = output.quantile(0.9).detach().cpu().numpy()
    _forecast_counter_add(batches=1, endpoints=int(len(contexts)))
    return median, q10, q90


def _load_or_build_forecasts(width_hours, horizon_days, force=False):
    steps = horizon_steps(horizon_days, width_hours)
    key = timestral_cache_key(width_hours, horizon_days)
    path = timestral_cache_path(width_hours, horizon_days)
    endpoint_ids = ENDPOINTS["endpoint_id"].tolist()
    if path.exists() and not force:
        with np.load(path, allow_pickle=False) as cached:
            metadata = json.loads(str(cached["metadata"].item()))
            stored_key = str(cached["cache_key"].item())
            stored_ids = [str(value) for value in cached["endpoint_ids"].tolist()]
            median = np.asarray(cached["median"], dtype=float)
            q10 = np.asarray(cached["q10"], dtype=float)
            q90 = np.asarray(cached["q90"], dtype=float)
        if stored_key != key:
            raise RuntimeError("Cache de forecast com cache_key incompatível.")
        if stored_ids != endpoint_ids:
            raise RuntimeError("Cache de forecast com endpoint_ids incompatíveis.")
        assert_forecast_cache_metadata(
            metadata, width_hours, horizon_days, key, endpoint_ids
        )
        median, q10, q90 = validate_forecast_arrays(
            median, q10, q90, len(endpoint_ids), steps
        )
        current = current_observed_matrix(width_hours, endpoint_ids)
        features = forecast_feature_frame(
            endpoint_ids, median, q10, q90, current, width_hours, horizon_days
        )
        validate_forecast_feature_frame(features)
        TIMESTRAL_CACHE_METADATA[key] = metadata
        print(f"Forecast {width_hours} h / {horizon_days} d: cache válido "
              f"({median.shape}); features recomputadas e validadas.")
        return features, metadata

    contexts = build_timestral_contexts(CLEAN[LABEL_ORDER[0]], ENDPOINTS, width_hours)
    forecaster = load_timestral_forecaster()
    batch_size = int(TIMESTRAL["batch_size"])
    if batch_size < 1:
        raise ValueError("batch_size deve ser >= 1.")
    medians = []
    q10s = []
    q90s = []
    for start in range(0, len(contexts), batch_size):
        batch = contexts[start:start + batch_size]
        m, a, b = _forecast_batch(forecaster, batch, steps)
        medians.append(m)
        q10s.append(a)
        q90s.append(b)
    median = np.concatenate(medians, axis=0)
    q10 = np.concatenate(q10s, axis=0)
    q90 = np.concatenate(q90s, axis=0)
    validate_forecast_arrays(median, q10, q90, len(endpoint_ids), steps)
    current = current_observed_matrix(width_hours, endpoint_ids)
    features = forecast_feature_frame(
        endpoint_ids, median, q10, q90, current, width_hours, horizon_days
    )
    actual_device = str(forecaster.device)
    metadata = forecast_cache_metadata(
        width_hours, horizon_days, key, endpoint_ids, actual_device
    )
    temporary = path.with_name(f".{path.stem}.tmp.npz")
    np.savez_compressed(
        temporary,
        median=median,
        q10=q10,
        q90=q90,
        endpoint_ids=np.array(endpoint_ids),
        cache_key=np.array(key),
        metadata=np.array(canonical_json(metadata)),
    )
    os.replace(temporary, path)
    TIMESTRAL_CACHE_METADATA[key] = metadata
    print(f"Forecast {width_hours} h / {horizon_days} d: construído e cacheado "
          f"({median.shape}).")
    return features, metadata


def get_timestral_feature_matrix(width_hours, horizon_days):
    width_hours = int(width_hours)
    horizon_days = float(horizon_days)
    key = (width_hours, horizon_days)
    if key in TIMESTRAL_FEATURE_MATRICES:
        return TIMESTRAL_FEATURE_MATRICES[key]
    observed = get_feature_matrix(width_hours)
    forecast_features, _ = _load_or_build_forecasts(width_hours, horizon_days)
    if observed.index.tolist() != forecast_features.index.tolist():
        raise RuntimeError("Endpoints do forecast e das features observadas divergem.")
    combined = observed.join(forecast_features, how="left")
    expected = list(feature_columns(width_hours, "summary", True)) + list(TIMESTRAL_FORECAST_COLUMNS)
    if combined.columns.tolist() != expected:
        raise RuntimeError("Ordem/forma das features Timestral mudou.")
    TIMESTRAL_FEATURE_MATRICES[key] = combined
    return combined


def forecast_plan():
    combos = set()
    if CONFIG["search"]["enabled"]:
        for bin_hours in CONFIG["search"]["bin_hours_levels"]:
            for horizon_days in CONFIG["search"]["horizon_days_levels"]:
                combos.add((int(bin_hours), float(horizon_days)))
    else:
        for head, presets in CONFIG["quick"]["presets"].items():
            for preset in presets:
                combos.add((int(preset["bin_hours"]),
                            float(preset.get("horizon_days",
                                             TIMESTRAL["default_horizon_days"]))))
    if CONFIG["ablation"]["enabled"]:
        combos.add((int(CONFIG["ablation"]["base_bin_hours"]),
                    float(CONFIG["ablation"]["base_horizon_days"])))
    if not combos:
        raise RuntimeError("Nenhum par grade/horizonte para forecast.")
    return sorted(combos)


def forecast_cost_preview():
    plan = forecast_plan()
    batch_size = int(TIMESTRAL["batch_size"])
    per_config = int(math.ceil(len(ENDPOINTS) / batch_size))
    cached = 0
    missing = 0
    for width_hours, horizon_days in plan:
        if timestral_cache_path(width_hours, horizon_days).exists():
            cached += 1
        else:
            missing += 1
    return {
        "unique_configs": [(w, h) for w, h in plan],
        "n_endpoints": int(len(ENDPOINTS)),
        "batch_size": batch_size,
        "batches_per_config": per_config,
        "cached_configs": int(cached),
        "missing_configs": int(missing),
        "expected_batches_missing": int(missing * per_config),
        "expected_endpoints_missing": int(missing * len(ENDPOINTS)),
        "expected_batches_total": int(len(plan) * per_config),
    }


def print_forecast_preview():
    preview = forecast_cost_preview()
    print("\nPREVIEW DE FORECAST (antes de qualquer download/inferência)")
    print("  pares únicos grade×horizonte:", preview["unique_configs"])
    print("  endpoints:", preview["n_endpoints"], "| batch:", preview["batch_size"])
    print("  configs em cache:", preview["cached_configs"],
          "| faltando:", preview["missing_configs"])
    print("  batches faltantes:", preview["expected_batches_missing"],
          "| endpoints faltantes:", preview["expected_endpoints_missing"])
    print("  batches se tudo fosse recalculado:", preview["expected_batches_total"])
    return preview


def run_forecast_smoke():
    if not TIMESTRAL["forecast_smoke"]:
        print("forecast_smoke=False: nenhum forecast real será executado.")
        return None
    width_hours = int(PROTOCOL["default_bin_hours"])
    horizon_days = float(TIMESTRAL["default_horizon_days"])
    steps = horizon_steps(horizon_days, width_hours)
    sample = ENDPOINTS.iloc[:2].reset_index(drop=True)
    contexts = build_timestral_contexts(CLEAN[LABEL_ORDER[0]], sample, width_hours)
    forecaster = load_timestral_forecaster()
    median, q10, q90 = _forecast_batch(forecaster, contexts, steps)
    validate_forecast_arrays(median, q10, q90, len(sample), steps)
    print("SMOKE FORECAST OK | context:", contexts.shape,
          "| median:", median.shape, "| q10/q90:", q10.shape, q90.shape,
          "| endpoints:", sample["endpoint_id"].tolist())
    return {"context_shape": contexts.shape, "median_shape": median.shape,
            "q10_shape": q10.shape, "q90_shape": q90.shape,
            "endpoint_ids": sample["endpoint_id"].tolist()}


def ensure_forecasts_for_plan():
    for width_hours, horizon_days in forecast_plan():
        get_timestral_feature_matrix(width_hours, horizon_days)


print("Forecast congelado pronto; features:",
      len(TIMESTRAL_FORECAST_COLUMNS), "| cache:",
      TIMESTRAL_CACHE_DIR)


## Heads supervisionados

Os únicos modelos ajustados neste notebook são **Ridge** e **ExtraTrees**. Não há média constante, SVR, Random Forest, TCN, ARIMA nem repetição dos baselines standalone.

Para cada head, rótulo e fold externo:

1. as features observadas e as features de forecast congeladas são montadas para todos os endpoints;
2. o imputador, a escala fixa de flags de ausência e o modelo são ajustados apenas no treino interno;
3. uma configuração é escolhida exclusivamente pela MAE macro por componente no único grupo de validação interna;
4. o modelo escolhido é refeito em todo o treino externo;
5. o teste externo é avaliado uma vez e exportado com o nome do pipeline (`timestral_ridge` / `timestral_extra_trees`).

Não há vencedor global entre Ridge e ExtraTrees. Cada head tem seu próprio resultado, e a comparação entre eles é descritiva. O teste externo nunca entra na seleção.

A semente é fixa por head e fold e é a mesma entre todos os candidatos daquele head. Os presets rápidos são pequenos e incluem alternativas de representação, pré-processamento e pesos; a busca Optuna opcional explora também grade e horizonte, sempre dentro do fold interno.


In [ ]:
# =============================================================================
# SEÇÃO 8 — IMPUTAÇÃO/ESCALA APRENDIDAS, CANDIDATOS E MODELOS
# =============================================================================

STRUCTURAL_KEYS = {"bin_hours", "representation", "preprocessing", "timing", "weights"}


# Pré-processador compartilhado com cer_rul (mesmo comportamento numérico:
# mediana do treino, fallback 0 para colunas totalmente ausentes, signed log
# antes do scaler e scaler ajustado somente no treino).
signed_log1p = _cer_rul.signed_log1p
TabularPreprocessor = _cer_rul.TabularPreprocessor


def _validate_param_range(name, value, lower, upper):
    if not isinstance(value, (int, float, np.integer, np.floating)):
        raise ValueError(f"{name} deve ser numérico.")
    value = float(value)
    if not np.isfinite(value) or not (lower <= value <= upper):
        raise ValueError(f"{name}={value} fora de [{lower}, {upper}].")
    return value


def validate_model_params(family, params):
    params = dict(params)
    if family == "mean":
        if params:
            raise ValueError("A referência de média não aceita parâmetros de modelo.")
        return {}
    if family == "ridge":
        if set(params) != {"alpha"}:
            raise ValueError("Ridge exige exatamente alpha.")
        _validate_param_range("alpha", params["alpha"], *MODEL_BOUNDS["ridge"]["alpha"])
        return {"alpha": float(params["alpha"])}
    if family == "svr":
        if set(params) != {"C", "epsilon", "gamma"}:
            raise ValueError("SVR exige C, epsilon e gamma.")
        _validate_param_range("C", params["C"], *MODEL_BOUNDS["svr"]["C"])
        _validate_param_range("epsilon", params["epsilon"],
                              *MODEL_BOUNDS["svr"]["epsilon"])
        gamma = params["gamma"]
        if gamma != "scale":
            _validate_param_range("gamma", gamma, 1e-9, 1e6)
            gamma = float(gamma)
        return {"C": float(params["C"]), "epsilon": float(params["epsilon"]),
                "gamma": gamma}
    if family == "extra_trees":
        if set(params) != {"max_depth", "min_samples_leaf", "max_features"}:
            raise ValueError("ExtraTrees exige max_depth, min_samples_leaf e max_features.")
        depth = int(params["max_depth"])
        leaf = int(params["min_samples_leaf"])
        if depth != params["max_depth"] or leaf != params["min_samples_leaf"]:
            raise ValueError("max_depth e min_samples_leaf devem ser inteiros.")
        _validate_param_range("max_depth", depth, *MODEL_BOUNDS["extra_trees"]["max_depth"])
        _validate_param_range("min_samples_leaf", leaf,
                              *MODEL_BOUNDS["extra_trees"]["min_samples_leaf"])
        max_features = params["max_features"]
        if max_features not in ("sqrt", 0.5, 1.0):
            raise ValueError("max_features deve ser 'sqrt', 0.5 ou 1.0.")
        return {"max_depth": depth, "min_samples_leaf": leaf,
                "max_features": max_features}
    raise ValueError(f"Família desconhecida: {family}")


def seed_for(fold_id, family):
    if family == "mean":
        return 0
    return int(sha256_json({
        "seed_base": int(CONFIG["seed_base"]),
        "fold": str(fold_id),
        "family": str(family),
    })[:8], 16) % (2 ** 31 - 1)


def candidate_id(candidate):
    payload = {key: candidate[key] for key in
               ("family", "params", "bin_hours", "representation",
                "preprocessing", "add_missing_indicators", "timing", "weights")}
    payload["origin"] = candidate.get("origin", "quick")
    return sha256_json(payload)[:12]


def candidate_from_preset(family, preset, fold_id, preset_index):
    preset = dict(preset)
    if family == "mean":
        if preset:
            raise ValueError("Preset da média deve ser vazio.")
        params = {}
        structure = {"bin_hours": int(PROTOCOL["default_bin_hours"]),
                     "representation": "current", "preprocessing": "none",
                     "add_missing_indicators": PREPROCESSING_CONFIG["add_missing_indicators"],
                     "timing": False, "weights": "none"}
    else:
        if "bin_hours" not in preset:
            raise ValueError("Preset sem bin_hours.")
        if "preprocessing" in preset:
            raise ValueError(
                "Preset não aceita 'preprocessing': o método vem de "
                "CONFIG['preprocessing']['method'] (um único controle)."
            )
        structure = {
            "bin_hours": int(preset["bin_hours"]),
            "representation": preset["representation"],
            "preprocessing": PREPROCESSING_CONFIG["method"],
            "add_missing_indicators": PREPROCESSING_CONFIG["add_missing_indicators"],
            "timing": bool(preset["timing"]),
            "weights": preset["weights"],
        }
        params = {key: value for key, value in preset.items()
                  if key not in STRUCTURAL_KEYS}
    params = validate_model_params(family, params)
    if structure["bin_hours"] not in PROTOCOL["bin_hours_candidates"]:
        raise ValueError("Largura de grade não aprovada.")
    if structure["representation"] not in REPRESENTATIONS:
        raise ValueError("Representação não aprovada.")
    if structure["preprocessing"] not in PREPROCESSING_OPTIONS:
        raise ValueError("Pré-processamento não aprovado.")
    if structure["weights"] not in WEIGHT_SCHEMES:
        raise ValueError("Esquema de pesos não aprovado.")
    candidate = {
        "family": family,
        "params": params,
        **structure,
        "seed": seed_for(fold_id, family),
        "origin": f"quick:{family}:{preset_index}",
        "fold_id": fold_id,
    }
    candidate["candidate_id"] = candidate_id(candidate)
    return candidate


def quick_candidates(fold_id):
    candidates = []
    for family in MODEL_FAMILIES:
        presets = CONFIG["quick"]["presets"].get(family, [])
        for preset_index, preset in enumerate(presets):
            candidates.append(
                candidate_from_preset(family, preset, fold_id, preset_index)
            )
    if not any(candidate["family"] == "mean" for candidate in candidates):
        raise ValueError("O baseline de média é obrigatório.")
    return candidates


class FittedModel:
    def __init__(self, candidate, preprocessor=None, estimator=None,
                 constant=None, fit_rows=None):
        self.candidate = candidate
        self.preprocessor = preprocessor
        self.estimator = estimator
        self.constant = constant
        self.fit_rows = fit_rows

    def predict(self, frame):
        if self.constant is not None:
            return np.full(len(frame), float(self.constant), dtype=float)
        matrix = self.preprocessor.transform(frame)
        return np.asarray(self.estimator.predict(matrix), dtype=float)


def fit_and_predict(candidate, X_train, y_train, X_eval, train_ids, train_phases,
                    counter=None, purpose="selection"):
    if counter is not None:
        counter["model_fits"] = counter.get("model_fits", 0) + 1
        counter[f"{purpose}_fits"] = counter.get(f"{purpose}_fits", 0) + 1
        if candidate["family"] == "mean":
            counter["baseline_fits"] = counter.get("baseline_fits", 0) + 1
            counter[f"{purpose}_baseline_fits"] = (
                counter.get(f"{purpose}_baseline_fits", 0) + 1
            )
    y_train = np.asarray(y_train, dtype=float)
    if len(y_train) == 0:
        raise ValueError("Treino vazio.")
    if not np.isfinite(y_train).all():
        raise ValueError("Alvo de treino não finito.")

    if candidate["family"] == "mean":
        constant = float(np.mean(y_train))  # sem pesos, por definição
        model = FittedModel(candidate, constant=constant, fit_rows=int(len(y_train)))
        return model.predict(X_eval), model, {"n_features_out": 0, "constant": constant}

    preprocessor = TabularPreprocessor(
        candidate["preprocessing"],
        add_missing_indicators=candidate["add_missing_indicators"],
    )
    matrix_train = preprocessor.fit_transform(X_train)
    matrix_eval = preprocessor.transform(X_eval)
    weights = compute_weights(train_ids, train_phases, candidate["weights"])
    family = candidate["family"]
    params = candidate["params"]
    if family == "ridge":
        estimator = Ridge(alpha=params["alpha"], fit_intercept=True)
    elif family == "svr":
        estimator = SVR(
            kernel="rbf", C=params["C"], epsilon=params["epsilon"],
            gamma=params["gamma"], cache_size=1024,
        )
    elif family == "extra_trees":
        estimator = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=int(candidate["seed"]),
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Família sem implementação: {family}")
    estimator.fit(matrix_train, y_train, sample_weight=weights)
    model = FittedModel(candidate, preprocessor=preprocessor, estimator=estimator,
                        constant=None, fit_rows=int(len(y_train)))
    prediction = model.predict(X_eval)
    metadata = {
        "n_features_in": int(len(preprocessor.columns_)),
        "n_features_out": int(preprocessor.n_features_out_),
        "n_missing_indicators": int(len(preprocessor.indicator_columns_)),
        "constant_fallback_columns": list(preprocessor.constant_fallback_columns_),
        "weight_min": float(np.min(weights)),
        "weight_max": float(np.max(weights)),
        "weight_mean": float(np.mean(weights)),
        "seed": int(candidate["seed"]),
    }
    return prediction, model, metadata


print("Modelos disponíveis:", MODEL_FAMILIES)
print("SVR:", SVR_EPSILON_UNITS_NOTE)


In [ ]:
# =============================================================================
# SEÇÃO 9 — PESOS, MÉTRICAS E AGREGAÇÃO MACRO ENTRE FOLDS
# =============================================================================


def compute_weights(ids, phases, scheme):
    return _cer_rul.compute_weights(ids, phases, scheme,
                                    weight_schemes=WEIGHT_SCHEMES)


regression_metrics = _cer_rul.regression_metrics
metric_breakdown = _cer_rul.metric_breakdown


def selection_objective(y_true_metric, y_pred_metric, ids):
    y_true_metric = np.asarray(y_true_metric, dtype=float)
    y_pred_metric = np.asarray(y_pred_metric, dtype=float)
    ids = np.asarray(ids, dtype=int)
    component_mae = {}
    for fault_id in sorted(set(int(value) for value in ids)):
        mask = ids == fault_id
        component_mae[str(fault_id)] = regression_metrics(
            y_true_metric[mask], y_pred_metric[mask])["mae"]
    score = float(np.mean(list(component_mae.values())))
    return score, component_mae


def scaled_prediction_to_units(prediction_scaled, label):
    return _cer_rul.scaled_prediction_to_units(
        prediction_scaled, label, label_meta=PROTOCOL["labels"])


def plot_metric_columns(label):
    """Colunas e unidade usadas nos gráficos: y_metric é dias ou pp."""
    if label not in PROTOCOL["labels"]:
        raise ValueError(f"Rótulo desconhecido: {label}")
    return "y_metric", "pred_metric", PROTOCOL["labels"][label]["unit"]


def predictions_frame(target_subset, prediction_scaled, label, fold, candidate):
    meta = PROTOCOL["labels"][label]
    if len(target_subset) != len(prediction_scaled):
        raise ValueError("Predição e alvo desalinhados.")
    native, metric = scaled_prediction_to_units(prediction_scaled, label)
    frame = pd.DataFrame({
        "label": label,
        "unit": meta["unit"],
        "fold_id": fold["fold_id"],
        "test_equipment": fold["test_equipment"],
        "endpoint_id": target_subset["endpoint_id"].tolist(),
        "Falha": target_subset["Falha"].astype(int).tolist(),
        "equipment": target_subset["equipment"].astype(str).tolist(),
        "Date": pd.to_datetime(target_subset["Date"]).tolist(),
        "phase": target_subset["phase"].astype(str).tolist(),
        "candidate_id": candidate["candidate_id"],
        "family": candidate["family"],
        "y_native": target_subset["RUL"].astype(float).to_numpy(),
        "pred_native": native,
        "y_metric": target_subset["y_metric"].astype(float).to_numpy(),
        "pred_metric": metric,
        "residual_metric": metric - target_subset["y_metric"].astype(float).to_numpy(),
        "split_role": "outer_test",
    })
    if not np.isfinite(frame["pred_metric"].to_numpy(float)).all():
        raise ValueError("Predição não finita; não há clamp.")
    return frame


def predictions_to_json(frame):
    records = []
    for row in frame.itertuples(index=False):
        records.append({
            "label": row.label,
            "unit": row.unit,
            "fold_id": row.fold_id,
            "test_equipment": row.test_equipment,
            "endpoint_id": row.endpoint_id,
            "Falha": int(row.Falha),
            "equipment": row.equipment,
            "Date": pd.Timestamp(row.Date).isoformat(),
            "phase": row.phase,
            "candidate_id": row.candidate_id,
            "family": row.family,
            "y_native": float(row.y_native),
            "pred_native": float(row.pred_native),
            "y_metric": float(row.y_metric),
            "pred_metric": float(row.pred_metric),
            "residual_metric": float(row.residual_metric),
            "split_role": row.split_role,
        })
    return records


def fold_metric_rows(prediction_frame):
    return _cer_rul.fold_metric_rows(
        prediction_frame, label_meta=PROTOCOL["labels"])


def aggregate_metric_rows(rows):
    """Wrapper: o módulo devolve a tag legada de 4 equipamentos; aqui a tag
    reflete a unidade e o número real de grupos do dataset selecionado."""
    result = _cer_rul.aggregate_metric_rows(rows)
    if not result.empty:
        result.loc[result["scope"] == "equipment_macro", "aggregation"] = (
            f"media_igual_entre_{len(EQUIPMENT_ORDER)}_{UNIT_LABEL_PLURAL}_por_familia"
        )
    return result




print("Funções de métricas prontas; bias = predição - real; sem clamp e sem MAPE.")


In [ ]:
# =============================================================================
# SEÇÃO 10 — IDENTIDADES, RESUME ATÔMICO E INTEGRIDADE DE PREDIÇÕES
# =============================================================================

class RunIntegrityError(RuntimeError):
    pass


EFFECTIVE_CONFIG = {
    "run_training": bool(CONFIG["run_training"]),
    "seed_base": int(CONFIG["seed_base"]),
    "quick": CONFIG["quick"],
    "search": CONFIG["search"],
    "ablation": CONFIG["ablation"],
    "dataset": DATASET_ID,
    "preprocessing": PREPROCESSING_CONFIG,
    "code_revision": CONFIG.get("code_revision", "rul-cpfl-baselines-v2"),
    "feature_code_version": FEATURE_CODE_VERSION,
    "n_estimators": N_ESTIMATORS,
    "model_bounds": MODEL_BOUNDS,
    "preprocessing_options": PREPROCESSING_OPTIONS,
    "weight_schemes": WEIGHT_SCHEMES,
    "model_families": MODEL_FAMILIES,
    "model_specific": CONFIG.get("model_specific", {}),
}
EFFECTIVE_CONFIG_HASH = sha256_json(EFFECTIVE_CONFIG)
PROTOCOL_HASH = sha256_json(PROTOCOL)
DATA_IDENTITY = {
    **DATA_BASE_IDENTITY,
    "endpoint_sha256": ENDPOINT_HASH,
    "split_sha256": SPLIT_HASH,
}
DATA_IDENTITY_HASH = sha256_json(DATA_IDENTITY)
RUN_ID = sha256_json({
    "effective_config": EFFECTIVE_CONFIG,
    "protocol": PROTOCOL,
    "data_identity": DATA_IDENTITY,
})[:16]
RUN_DIR = RESULTS_DIR / RUN_ID
RUNS_DIR = RUN_DIR / "runs"
ABLATION_DIR = RUN_DIR / "ablation"
PLOTS_DIR = RUN_DIR / "plots"

RUN_REQUIRED_KEYS = (
    "run_identity", "label", "family", "fold_id", "candidate_mode",
    "chosen_candidate", "metrics", "counts", "predictions",
)


def run_identity_for(label, fold, family, candidate_mode, expected_prediction_keys):
    return sha256_json({
        "run_id": RUN_ID,
        "label": label,
        "family": family,
        "fold_id": fold["fold_id"],
        "candidate_mode": candidate_mode,
        "test_equipment": fold["test_equipment"],
        "prediction_keys_sha256": sha256_json(sorted(expected_prediction_keys)),
    })


def run_file_path(label, fold_id, family):
    return RUNS_DIR / f"{label}__{fold_id}__{family}.json"


def seal_payload(payload):
    sealed = json.loads(canonical_json(payload))
    sealed["complete"] = True
    sealed.pop("integrity_sha256", None)
    sealed["integrity_sha256"] = sha256_json(sealed)
    return sealed


def save_complete_run(path, payload):
    sealed = seal_payload(payload)
    if not sealed.get("predictions"):
        raise RunIntegrityError("Recusando gravar execução sem predições.")
    atomic_write_json(path, sealed)
    return sealed


def load_complete_run(path, expected_identity, expected_prediction_keys):
    path = Path(path)
    if not path.exists():
        return None
    payload = read_json(path)
    for key in RUN_REQUIRED_KEYS:
        if key not in payload:
            raise RunIntegrityError(f"Execução incompleta, campo ausente: {key}")
    if payload.get("complete") is not True:
        raise RunIntegrityError("Execução sem marcador complete=True.")
    if "predictions" not in payload or not isinstance(payload["predictions"], list):
        raise RunIntegrityError("Execução completa sem lista de predições.")
    stored_integrity = payload.get("integrity_sha256")
    recomputed = dict(payload)
    recomputed.pop("integrity_sha256", None)
    if stored_integrity != sha256_json(recomputed):
        raise RunIntegrityError("Falha de integridade do JSON de execução.")
    if payload.get("run_identity") != expected_identity:
        raise RunIntegrityError(
            "Identidade incompatível para o mesmo caminho de execução."
        )
    prediction_ids = [record.get("endpoint_id") for record in payload["predictions"]]
    if any(identifier is None for identifier in prediction_ids):
        raise RunIntegrityError("Predição sem endpoint_id.")
    if len(prediction_ids) != len(set(prediction_ids)):
        raise RunIntegrityError("Predições duplicadas no JSON.")
    if set(prediction_ids) != set(expected_prediction_keys):
        raise RunIntegrityError(
            "Predições ausentes/extras em relação ao teste externo esperado."
        )
    return payload


print("RUN_ID:", RUN_ID)
print("Config efetiva:", EFFECTIVE_CONFIG_HASH[:16],
      "| dados:", DATA_IDENTITY_HASH[:16])


In [ ]:
# =============================================================================
# SEÇÃO 11 — BUSCA OPTUNA OPCIONAL (15 TENTATIVAS TOTAIS POR ESTUDO)
# Escopo: head × rótulo × fold externo. A grade/horizonte é escolhida no
# fold interno; nenhum trial vê o teste externo.
# =============================================================================

SEARCH_AXES = ("bin_hours_levels", "horizon_days_levels",
               "representation_levels", "preprocessing_levels",
               "timing_levels", "weights_levels")


def search_signature(label, fold, head):
    cfg = CONFIG["search"]
    payload = {
        "run_id": RUN_ID,
        "label": label,
        "fold": fold,
        "head": head,
        "pipeline_id": PIPELINE_BY_HEAD[head],
        "n_trials": int(cfg["n_trials"]),
        "families": list(cfg["families"]),
        "axes": {key: cfg[key] for key in SEARCH_AXES},
        "spaces": cfg["spaces"],
        "model_bounds": MODEL_BOUNDS,
        "n_estimators": N_ESTIMATORS,
        "dataset_id": DATASET_ID,
        "preprocessing": PREPROCESSING_CONFIG,
        "feature_code_version": FEATURE_CODE_VERSION,
        "forecast_feature_version": TIMESTRAL["forecast_feature_version"],
        "model_specific": TIMESTRAL,
        "sensor_sha256": SENSOR_HASH,
        "endpoint_hash": ENDPOINT_HASH,
        "data_identity_hash": DATA_IDENTITY_HASH,
        "sampler_seed": int(cfg["sampler_seed"]) + 1000 * int(fold["fold_index"]),
        "family_seed": seed_for(fold["fold_id"], head),
        "protocol_revision": PROTOCOL["revision"],
    }
    return sha256_json(payload), payload


def search_study_path_and_id(label, fold, head):
    signature_hash, _ = search_signature(label, fold, head)
    study_id = f"timestral-{signature_hash[:12]}"
    return STUDIES_DIR / f"{study_id}.sqlite3", study_id, signature_hash


def _load_or_create_study(label, fold, head):
    import optuna
    signature_hash, signature_payload = search_signature(label, fold, head)
    storage_path, study_id, _ = search_study_path_and_id(label, fold, head)
    budget = int(CONFIG["search"]["n_trials"])
    sampler = optuna.samplers.TPESampler(
        seed=int(signature_payload["sampler_seed"]),
        n_startup_trials=min(5, budget),
    )
    storage = "sqlite:///" + str(storage_path)
    if storage_path.exists():
        study = optuna.load_study(study_name=study_id, storage=storage,
                                  sampler=sampler)
        if study.user_attrs.get("signature_hash") != signature_hash:
            raise RuntimeError("Estudo existente com assinatura incompatível.")
        if int(study.user_attrs.get("budget", -1)) != budget:
            raise RuntimeError("Estudo existente com orçamento incompatível.")
        if study.user_attrs.get("data_identity_hash") != DATA_IDENTITY_HASH:
            raise RuntimeError("Estudo existente pertence a outros dados.")
        if study.user_attrs.get("protocol_revision") != PROTOCOL["revision"]:
            raise RuntimeError("Estudo existente pertence a outro protocolo.")
    else:
        study = optuna.create_study(
            study_name=study_id, storage=storage, direction="minimize",
            sampler=sampler,
        )
        study.set_user_attr("signature_hash", signature_hash)
        study.set_user_attr("signature_payload", signature_payload)
        study.set_user_attr("budget", budget)
        study.set_user_attr("data_identity_hash", DATA_IDENTITY_HASH)
        study.set_user_attr("protocol_revision", PROTOCOL["revision"])
        study.set_user_attr("head", head)
        study.set_user_attr("label", label)
        study.set_user_attr("fold_id", fold["fold_id"])
    if len(study.trials) > budget:
        raise RuntimeError("Estudo com mais tentativas que o orçamento declarado.")
    return study, study_id, signature_hash


def search_remaining_trials(label, fold, head):
    storage_path, study_id, signature_hash = search_study_path_and_id(
        label, fold, head)
    if not storage_path.exists():
        # O smoke/preview de dados não deve exigir Optuna quando não há estudo.
        return int(CONFIG["search"]["n_trials"]), 0
    try:
        import optuna
    except Exception as exc:
        raise RuntimeError(
            "Há um estudo Optuna existente, mas optuna não está instalado. "
            "Instale/ative a dependência opcional para validar a retomada; "
            "o smoke de dados sem estudo existente não exige optuna."
        ) from exc
    study = optuna.load_study(study_name=study_id,
                              storage="sqlite:///" + str(storage_path))
    if study.user_attrs.get("signature_hash") != signature_hash:
        raise RuntimeError("Estudo existente com assinatura incompatível.")
    existing = len(study.trials)
    budget = int(CONFIG["search"]["n_trials"])
    if existing > budget:
        raise RuntimeError("Estudo com mais tentativas que o orçamento.")
    return budget - existing, existing


def _suggest_search_candidate(trial, head, fold):
    cfg = CONFIG["search"]
    spaces = cfg["spaces"]
    bin_hours = int(trial.suggest_categorical("bin_hours", list(cfg["bin_hours_levels"])))
    horizon_days = float(trial.suggest_categorical(
        "horizon_days", list(cfg["horizon_days_levels"])))
    representation = trial.suggest_categorical(
        "representation", list(cfg["representation_levels"]))
    preprocessing = trial.suggest_categorical(
        "preprocessing", list(cfg["preprocessing_levels"]))
    timing = bool(trial.suggest_categorical("timing", list(cfg["timing_levels"])))
    weights = trial.suggest_categorical("weights", list(cfg["weights_levels"]))
    if head == "ridge":
        params = {"alpha": trial.suggest_categorical("alpha", spaces["ridge"]["alpha"])}
    elif head == "extra_trees":
        params = {
            "max_depth": trial.suggest_categorical(
                "max_depth", spaces["extra_trees"]["max_depth"]),
            "min_samples_leaf": trial.suggest_categorical(
                "min_samples_leaf", spaces["extra_trees"]["min_samples_leaf"]),
            "max_features": trial.suggest_categorical(
                "max_features", spaces["extra_trees"]["max_features"]),
        }
    else:
        raise ValueError(f"Head de busca inválido: {head}")
    params = validate_model_params(head, params)
    candidate = {
        "family": head,
        "params": params,
        "bin_hours": bin_hours,
        "horizon_days": horizon_days,
        "representation": representation,
        "preprocessing": preprocessing,
        "add_missing_indicators": PREPROCESSING_CONFIG["add_missing_indicators"],
        "timing": timing,
        "weights": weights,
        "seed": seed_for(fold["fold_id"], head),
        "origin": f"search:{head}:trial-{trial.number}",
        "fold_id": fold["fold_id"],
        "pipeline_id": PIPELINE_BY_HEAD[head],
    }
    candidate["candidate_id"] = timestral_candidate_id(candidate)
    trial.set_user_attr("candidate", candidate)
    trial.set_user_attr("seed", int(candidate["seed"]))
    return candidate


def run_search_study(label, fold, head, target, get_features, masks, counter):
    study, study_id, signature_hash = _load_or_create_study(label, fold, head)
    budget = int(CONFIG["search"]["n_trials"])
    trials_before = len(study.trials)
    remaining = budget - trials_before
    meta = PROTOCOL["labels"][label]
    target_train = target.loc[masks["inner_train"]]
    target_val = target.loc[masks["val"]]

    def objective(trial):
        candidate = _suggest_search_candidate(trial, head, fold)
        features = get_features(candidate["bin_hours"], candidate["horizon_days"])
        columns = list(timestral_feature_columns(
            candidate["bin_hours"], candidate["representation"], candidate["timing"]))
        x_train = features.loc[target_train["endpoint_id"], columns]
        x_val = features.loc[target_val["endpoint_id"], columns]
        prediction_scaled, _, fit_metadata = fit_and_predict(
            candidate,
            x_train,
            target_train["y_scaled"].to_numpy(float),
            x_val,
            target_train["Falha"].to_numpy(int),
            target_train["phase"].to_numpy(str),
            counter=counter,
            purpose="selection",
        )
        prediction_metric = (prediction_scaled * float(meta["divisor"])
                             * float(meta["metric_factor"]))
        score, component_mae = selection_objective(
            target_val["y_metric"].to_numpy(float),
            prediction_metric,
            target_val["Falha"].to_numpy(int),
        )
        trial.set_user_attr("selection_macro_component_mae", float(score))
        trial.set_user_attr("component_mae", component_mae)
        trial.set_user_attr("n_train", int(len(target_train)))
        trial.set_user_attr("n_val", int(len(target_val)))
        trial.set_user_attr("fit_metadata", fit_metadata)
        return float(score)

    if remaining > 0:
        print(f"Optuna {label}/{fold['fold_id']}/{head}: "
              f"{trials_before} tentativas existentes, {remaining} novas, "
              f"orçamento total={budget}")
        study.optimize(objective, n_trials=remaining)
    else:
        print(f"Optuna {label}/{fold['fold_id']}/{head}: orçamento completo "
              f"({trials_before}/{budget}); nenhuma tentativa nova.")

    complete = [trial for trial in study.trials
                if trial.state.name == "COMPLETE" and trial.value is not None]
    if not complete:
        raise RuntimeError(f"Estudo sem tentativa concluída: {study_id}")
    best = study.best_trial
    if "candidate" not in best.user_attrs:
        raise RuntimeError("Melhor tentativa sem candidato.")
    best_candidate = dict(best.user_attrs["candidate"])
    if best_candidate["family"] != head:
        raise RuntimeError("Melhor tentativa pertence a outro head.")
    trial_rows = []
    for trial in study.trials:
        candidate = trial.user_attrs.get("candidate")
        trial_rows.append({
            "label": label,
            "fold_id": fold["fold_id"],
            "head": head,
            "family": PIPELINE_BY_HEAD[head],
            "study_id": study_id,
            "trial_number": int(trial.number),
            "state": trial.state.name,
            "value": trial.value,
            "candidate": candidate,  # estruturado; serializado uma vez na exportação
            "component_mae": trial.user_attrs.get("component_mae", {}),
            "seed": trial.user_attrs.get("seed"),
        })
    return {
        "head": head,
        "family": PIPELINE_BY_HEAD[head],
        "study_id": study_id,
        "budget_total": budget,
        "trials_before": trials_before,
        "trials_after": len(study.trials),
        "new_trials": remaining,
        "best_trial": int(best.number),
        "best_score": float(best.value),
        "candidate": best_candidate,
        "trials": trial_rows,
        "signature_hash": signature_hash,
    }


print("Busca Timestral pronta (desabilitada por padrão).")


In [ ]:
# =============================================================================
# SEÇÃO 12 — HEADS RIDGE/EXTRATREES: SELEÇÃO INTERNA, REFIT E TESTE EXTERNO
# Cada head é um resultado independente; não há vencedor global.
# =============================================================================


def timestral_candidate_id(candidate):
    payload = {
        "family": candidate["family"],
        "pipeline_id": candidate["pipeline_id"],
        "params": candidate["params"],
        "bin_hours": int(candidate["bin_hours"]),
        "horizon_days": float(candidate["horizon_days"]),
        "representation": candidate["representation"],
        "preprocessing": candidate["preprocessing"],
        "add_missing_indicators": bool(candidate["add_missing_indicators"]),
        "timing": bool(candidate["timing"]),
        "weights": candidate["weights"],
        "origin": candidate.get("origin", "quick"),
    }
    return sha256_json(payload)[:12]


def finalize_timestral_candidate(candidate, head, horizon_days):
    candidate = dict(candidate)
    candidate["horizon_days"] = float(horizon_days)
    candidate["pipeline_id"] = PIPELINE_BY_HEAD[head]
    candidate["candidate_id"] = timestral_candidate_id(candidate)
    return candidate


def quick_head_candidates(fold_id, head):
    candidates = []
    for preset_index, preset in enumerate(CONFIG["quick"]["presets"].get(head, [])):
        core = {key: value for key, value in preset.items()
                if key != "horizon_days"}
        base = candidate_from_preset(head, core, fold_id, preset_index)
        candidates.append(finalize_timestral_candidate(
            base, head, preset.get("horizon_days",
                                   TIMESTRAL["default_horizon_days"])))
    if not candidates:
        raise RuntimeError(f"Sem presets quick para {head}.")
    return candidates


def masks_for_fold(target, fold):
    return _cer_rul.masks_for_fold(
        target, fold,
        id_to_equipment=ID_TO_EQUIPMENT,
        equipment_order=EQUIPMENT_ORDER,
    )


def evaluate_head_candidate_inner(candidate, target, masks, get_features, counter):
    meta = PROTOCOL["labels"][target["label"].iloc[0]]
    train = target.loc[masks["inner_train"]]
    validation = target.loc[masks["val"]]
    features = get_features(candidate["bin_hours"], candidate["horizon_days"])
    columns = list(timestral_feature_columns(
        candidate["bin_hours"], candidate["representation"], candidate["timing"]))
    x_train = features.loc[train["endpoint_id"], columns]
    x_val = features.loc[validation["endpoint_id"], columns]
    prediction_scaled, _, fit_metadata = fit_and_predict(
        candidate,
        x_train,
        train["y_scaled"].to_numpy(float),
        x_val,
        train["Falha"].to_numpy(int),
        train["phase"].to_numpy(str),
        counter=counter,
        purpose="selection",
    )
    prediction_metric = (prediction_scaled * float(meta["divisor"])
                         * float(meta["metric_factor"]))
    score, component_mae = selection_objective(
        validation["y_metric"].to_numpy(float),
        prediction_metric,
        validation["Falha"].to_numpy(int),
    )
    return {
        "candidate": candidate,
        "score": float(score),
        "component_mae": component_mae,
        "n_train": int(len(train)),
        "n_val": int(len(validation)),
        "fit_metadata": fit_metadata,
        "selection_source": "inner_fit",
    }


def select_head_inner(label, fold, head, target, masks, get_features, counter):
    mode = "search" if CONFIG["search"]["enabled"] else "quick"
    if head not in PIPELINE_BY_HEAD:
        raise ValueError(f"Head desconhecido: {head}")
    search_results = []
    search_trials = []
    if mode == "quick":
        candidates = quick_head_candidates(fold["fold_id"], head)
        records = [evaluate_head_candidate_inner(
            candidate, target, masks, get_features, counter)
            for candidate in candidates]
    else:
        result = run_search_study(label, fold, head, target, get_features,
                                  masks, counter)
        best_row = next(
            row for row in result["trials"]
            if int(row["trial_number"]) == int(result["best_trial"])
        )
        search_results.append({key: result[key] for key in
                               ("head", "family", "study_id", "budget_total",
                                "trials_before", "trials_after", "new_trials",
                                "best_trial", "best_score", "candidate")})
        search_trials.extend(result["trials"])
        records = [{
            "candidate": result["candidate"],
            "score": float(result["best_score"]),
            "component_mae": best_row.get("component_mae"),
            "n_train": int(masks["inner_train"].sum()),
            "n_val": int(masks["val"].sum()),
            "fit_metadata": None,
            "selection_source": f"optuna_best/{result['study_id']}",
        }]
    if not records:
        raise RuntimeError(f"Nenhum candidato interno para {head}.")
    chosen_record = min(
        records,
        key=lambda record: (float(record["score"]),
                            record["candidate"]["candidate_id"]),
    )
    chosen = chosen_record["candidate"]
    if chosen["family"] != head:
        raise AssertionError("A seleção interna misturou heads.")
    return {
        "mode": mode,
        "head": head,
        "pipeline_id": PIPELINE_BY_HEAD[head],
        "chosen": chosen,
        "chosen_record": chosen_record,
        "records": records,
        "search_results": search_results,
        "search_trials": search_trials,
    }


def _selected_head_predict(candidate, target, masks, get_features, counter,
                           purpose="final"):
    train = target.loc[masks["outer_train"]]
    test = target.loc[masks["test"]]
    features = get_features(candidate["bin_hours"], candidate["horizon_days"])
    columns = list(timestral_feature_columns(
        candidate["bin_hours"], candidate["representation"], candidate["timing"]))
    x_train = features.loc[train["endpoint_id"], columns]
    x_test = features.loc[test["endpoint_id"], columns]
    prediction_scaled, fitted, fit_metadata = fit_and_predict(
        candidate,
        x_train,
        train["y_scaled"].to_numpy(float),
        x_test,
        train["Falha"].to_numpy(int),
        train["phase"].to_numpy(str),
        counter=counter,
        purpose=purpose,
    )
    return test, prediction_scaled, fitted, fit_metadata


def execute_label_fold_head(label, fold, family, get_features, counter):
    if family not in MODEL_FAMILIES:
        raise ValueError(f"Pipeline desconhecida: {family}")
    head = HEAD_BY_PIPELINE[family]
    target = TARGETS[label]
    masks = masks_for_fold(target, fold)
    expected_keys = target.loc[masks["test"], "endpoint_id"].tolist()
    candidate_mode = "search" if CONFIG["search"]["enabled"] else "quick"
    identity = run_identity_for(label, fold, family, candidate_mode, expected_keys)
    path = run_file_path(label, fold["fold_id"], family)
    existing = load_complete_run(path, identity, expected_keys)
    if existing is not None:
        print(f"{label}/{fold['fold_id']}/{family}: execução completa reutilizada.")
        return existing, True

    counter_before = dict(counter)
    forecast_before = dict(FORECAST_COUNTER)
    selection = select_head_inner(label, fold, head, target, masks,
                                  get_features, counter)
    chosen = selection["chosen"]
    test, prediction_scaled, fitted, fit_metadata = _selected_head_predict(
        chosen, target, masks, get_features, counter
    )
    prediction_frame = predictions_frame(test, prediction_scaled, label, fold, chosen)
    # O nome do pipeline distingue estes resultados dos baselines standalone.
    prediction_frame["family"] = family
    prediction_frame["head"] = head
    if len(prediction_frame) != len(expected_keys):
        raise RuntimeError("Predições ausentes antes de gravar o JSON.")
    meta = PROTOCOL["labels"][label]
    metrics = metric_breakdown(
        prediction_frame["y_metric"].to_numpy(float),
        prediction_frame["pred_metric"].to_numpy(float),
        prediction_frame["Falha"].to_numpy(int),
        prediction_frame["equipment"].to_numpy(str),
        prediction_frame["phase"].to_numpy(str),
        meta["unit"],
    )
    fit_delta = {key: int(counter.get(key, 0) - counter_before.get(key, 0))
                 for key in sorted(counter)}
    forecast_delta = {key: int(FORECAST_COUNTER.get(key, 0) - forecast_before.get(key, 0))
                      for key in sorted(FORECAST_COUNTER)}
    payload = {
        "run_identity": identity,
        "run_id": RUN_ID,
        "label": label,
        "family": family,
        "head": head,
        "pipeline_id": family,
        "unit": meta["unit"],
        "divisor": float(meta["divisor"]),
        "metric_factor": float(meta["metric_factor"]),
        "fold_id": fold["fold_id"],
        "fold": {key: (list(value) if isinstance(value, tuple) else value)
                 for key, value in fold.items()},
        "candidate_mode": selection["mode"],
        "chosen_candidate": chosen,
        "chosen_record": {key: value for key, value in selection["chosen_record"].items()
                          if key != "candidate"},
        "selection_records": selection["records"],
        "search_results": selection["search_results"],
        "search_trials": selection["search_trials"],
        "fit_metadata": fit_metadata,
        "fit_counts_this_run": fit_delta,
        "forecast_counts_this_run": forecast_delta,
        "forecast_cache_key": timestral_cache_key(
            chosen["bin_hours"], chosen["horizon_days"]),
        "forecast_identity": timestral_cache_identity_payload(
            chosen["bin_hours"], chosen["horizon_days"]),
        "counts": {
            "n_endpoints": int(len(target)),
            "n_inner_train": int(masks["inner_train"].sum()),
            "n_val": int(masks["val"].sum()),
            "n_outer_train": int(masks["outer_train"].sum()),
            "n_outer_test": int(masks["test"].sum()),
        },
        "metrics": metrics,
        "predictions": predictions_to_json(prediction_frame),
    }
    save_complete_run(path, payload)
    print(f"{label}/{fold['fold_id']}/{family}: escolhido "
          f"{chosen['candidate_id']} (head={head}, h={chosen['horizon_days']}d, "
          f"bin={chosen['bin_hours']}h), teste={len(prediction_frame)}, "
          f"MAE={metrics['overall']['mae']:.4f} {meta['unit']}")
    return payload, False


def expected_run_count():
    return int(len(LABEL_ORDER) * len(SPLITS) * len(MODEL_FAMILIES))


def completed_run_payload(label, fold, family):
    target = TARGETS[label]
    masks = masks_for_fold(target, fold)
    expected_keys = target.loc[masks["test"], "endpoint_id"].tolist()
    candidate_mode = "search" if CONFIG["search"]["enabled"] else "quick"
    identity = run_identity_for(label, fold, family, candidate_mode, expected_keys)
    path = run_file_path(label, fold["fold_id"], family)
    if not path.exists():
        return None
    return load_complete_run(path, identity, expected_keys)


def estimate_expected_fits():
    mode = "search" if CONFIG["search"]["enabled"] else "quick"
    selection_fits = 0
    final_fits = 0
    for label in LABEL_ORDER:
        for fold in SPLITS.to_dict(orient="records"):
            for family in MODEL_FAMILIES:
                if completed_run_payload(label, fold, family) is not None:
                    continue
                final_fits += 1
                head = HEAD_BY_PIPELINE[family]
                if mode == "quick":
                    selection_fits += len(quick_head_candidates(fold["fold_id"], head))
                else:
                    remaining, _ = search_remaining_trials(label, fold, head)
                    selection_fits += remaining
    ablation_fits = 0
    if CONFIG["ablation"]["enabled"]:
        ablation_fits = len(CONFIG["ablation"]["levels"]) * len(LABEL_ORDER) * len(SPLITS)
    return {
        "mode": mode,
        "expected_runs": expected_run_count(),
        "selection_fits": int(selection_fits),
        "final_fits": int(final_fits),
        "ablation_fits": int(ablation_fits),
        "total_model_fits": int(selection_fits + final_fits + ablation_fits),
        "no_mean_or_svr": True,
    }


def print_fit_preview(preview):
    print("\nPREVIEW DE AJUSTES SUPERVISIONADOS (antes de qualquer treino)")
    print("  modo:", preview["mode"])
    print("  execuções label×fold×head esperadas:", preview["expected_runs"])
    print("  seleção interna:", preview["selection_fits"])
    print("  refits finais:", preview["final_fits"])
    print("  ablação:", preview["ablation_fits"])
    print("  TOTAL de ajustes:", preview["total_model_fits"])
    print("  JSON completo no disco é reutilizado e não conta.")


print("Heads Timestral prontos:", MODEL_FAMILIES)


## Testes leves de runtime

Esta seção não ajusta modelos. Os testes cobrem:

- média de duplicatas com multiplicidade e conflito de alvo (média de **todas** as linhas, não dos valores únicos);
- conflito de sensores na mesma chave, que deve falhar;
- alinhamento bruto/limpo dos rótulos do dataset selecionado;
- inclusão obrigatória do endpoint final;
- bin `(left, right]`, ancoragem à direita e imutabilidade do contexto ao acrescentar uma linha futura;
- inclinação por dia real;
- splits disjuntos;
- imputador/scaler ajustados só no treino, incluindo coluna totalmente vazia;
- pesos normalizados;
- unidades nativas, inclusive predição negativa sem clamp;
- fingerprint determinístico e rejeição de JSON de retomada sem predições.


In [ ]:
# =============================================================================
# SEÇÃO 15 — TESTES LEVES (SEM AJUSTE PESADO)
# =============================================================================


def _synthetic_frame(n=40, fault_id=1):
    dates = pd.date_range("2024-01-01", periods=n, freq="h")
    frame = pd.DataFrame({
        "Date": dates,
        "Falha": [fault_id] * n,
        "RUL": np.linspace(80.0, 0.0, n),
        "raw_count": [1] * n,
    })
    for position, column in enumerate(SENSOR_COLUMNS):
        frame[column] = np.linspace(1.0 + position, 2.0 + position, n)
    return frame.loc[:, ["Date", "Falha", "RUL", "raw_count"] + list(SENSOR_COLUMNS)]


def run_self_tests():
    results = []

    def check(name, condition):
        if not condition:
            raise AssertionError(name)
        results.append(name)

    # 1. Consolidação: média de todas as linhas, inclusive repetidas.
    base_date = pd.Timestamp("2024-01-01 00:00:00")
    raw = pd.DataFrame({
        "Date": [base_date, base_date, base_date, base_date + pd.Timedelta(hours=1)],
        "Falha": [1, 1, 1, 1],
        "RUL": [1.0, 1.0, 0.0, 5.0],
    })
    for position, column in enumerate(SENSOR_COLUMNS):
        raw[column] = [1.0 + position, 1.0 + position, 1.0 + position,
                       1.01 + position]
    raw = raw.loc[:, list(CSV_COLUMNS)]
    clean, audit, _ = consolidate_frame(raw, "synthetic", enforce_expectations=False)
    first_target = float(clean.loc[clean["Date"] == base_date, "RUL"].iloc[0])
    check("duplicate_mean_all_rows", abs(first_target - (2.0 / 3.0)) < 1e-12)
    check("duplicate_mean_not_unique_values", abs(first_target - 0.5) > 1e-6)
    check("target_conflict_audit", audit["target_conflict_keys"] == 1)
    check("multiplicity_audit", audit["raw_max_multiplicity"] == 3)
    check("fault_ids_exact_integer",
          coerce_fault_ids(["1", "7"]).tolist() == [1, 7])
    try:
        coerce_fault_ids(["1", "1.9"])
        raise AssertionError("Falha fracionária deveria falhar")
    except ValueError:
        results.append("fault_ids_fractional_rejected")
    clean_fp = clean_frame_fingerprint(clean)
    check("clean_fingerprint_stable", clean_fp == clean_frame_fingerprint(clean))
    changed_clean = clean.copy()
    changed_clean.loc[0, "RUL"] = changed_clean.loc[0, "RUL"] + 0.01
    check("clean_fingerprint_sensitive",
          clean_fp != clean_frame_fingerprint(changed_clean))

    conflicting = raw.copy()
    conflicting.loc[1, SENSOR_COLUMNS[0]] = 999.0
    try:
        consolidate_frame(conflicting, "synthetic", enforce_expectations=False)
        raise AssertionError("feature conflict deveria falhar")
    except ValueError:
        results.append("feature_conflict_fails")

    # 2. Alinhamento bruto/limpo dos rótulos do dataset (sintético).
    raw_map = {}
    clean_map = {}
    for label in LABEL_ORDER:
        frame = raw.copy()
        frame["RUL"] = frame["RUL"] + LABEL_ORDER.index(label)
        raw_map[label] = frame
        clean_map[label] = _synthetic_frame(n=4)
    check("raw_alignment", assert_raw_alignment(raw_map))
    check("clean_alignment", assert_clean_alignment(clean_map))

    # 3. Endpoint final sempre incluído, e nenhum dentro do aquecimento.
    endpoint_reference = _synthetic_frame(n=40)
    endpoints = build_endpoint_rows(endpoint_reference, warmup=5, stride=7,
                                    enforce=False)
    for fault_id, group in endpoints.groupby("Falha", sort=True):
        check(f"endpoint_first_after_warmup_{fault_id}", group["record_index"].min() >= 5)
        check(f"endpoint_final_{fault_id}", group["record_index"].max() == 39)
        check(f"endpoint_no_after_final_{fault_id}",
              group["n_records_after"].min() == 0)

    # 4. Bin (left, right] e ancoragem à direita.
    endpoint_date = pd.Timestamp("2024-01-03 12:00:00")
    sparse_dates = pd.Series([
        endpoint_date - pd.Timedelta(hours=13),
        endpoint_date - pd.Timedelta(hours=12),  # exatamente na borda: entra no bin anterior
        endpoint_date - pd.Timedelta(hours=11),
        endpoint_date,
    ])
    sparse_values = np.vstack([np.ones(8) * value for value in (10, 20, 30, 40)])
    last_values, last_times, counts = _bin_stats(
        sparse_dates, sparse_values, endpoint_date, 12)
    check("bin_left_boundary_excluded_from_right", counts[30] == 2 and counts[31] == 2)
    check("bin_last_value_causal", last_values[30, 0] == 20 and last_values[31, 0] == 40)
    check("bin_last_time_causal",
          last_times.iloc[30] == endpoint_date - pd.Timedelta(hours=12)
          and last_times.iloc[31] == endpoint_date)

    def same_features(first, second):
        return all(
            (math.isnan(first[key]) and math.isnan(second[key]))
            or abs(first[key] - second[key]) < 1e-12
            for key in first
        )

    # 5. Acrescentar uma linha futura não altera o contexto já construído.
    group = _synthetic_frame(n=80)
    endpoint_row = type("Endpoint", (), {
        "record_index": 60, "Date": group.loc[60, "Date"]
    })()
    before = build_feature_row(endpoint_row, group, 12)
    future = group.iloc[:61].copy()
    future_row = future.iloc[[-1]].copy()
    future_row["Date"] = future["Date"].max() + pd.Timedelta(hours=1)
    for column in SENSOR_COLUMNS:
        future_row[column] = future_row[column] + 1000.0
    future_row["Falha"] = 1
    future_row["RUL"] = 0.0
    future_row["raw_count"] = 1
    after = build_feature_row(endpoint_row, pd.concat([group, future_row],
                                                      ignore_index=True), 12)
    check("future_row_does_not_change_past_context", same_features(before, after))

    # 5b. Registros antigos fora da grade não influenciam valor/resumo da janela.
    long_group = _synthetic_frame(n=600)
    long_group.loc[:199, list(SENSOR_COLUMNS)] += 10000.0
    long_endpoint = type("Endpoint", (), {
        "record_index": 590, "Date": long_group.loc[590, "Date"]
    })()
    full_window = build_feature_row(long_endpoint, long_group, 12)
    trimmed = long_group.iloc[200:].reset_index(drop=True)
    trimmed_endpoint = type("Endpoint", (), {
        "record_index": 390, "Date": long_group.loc[590, "Date"]
    })()
    trimmed_window = build_feature_row(trimmed_endpoint, trimmed, 12)
    check("old_out_of_grid_rows_do_not_change_window",
          same_features(full_window, trimmed_window))

    # 5c. A representação current não contém valores brutos de bins nem resumo.
    current_columns = feature_columns(12, "current", True)
    summary_columns = feature_columns(12, "summary", True)
    check("current_has_no_raw_bins",
          not any(column.startswith("bin") or "__last__" in column
                  for column in current_columns))
    check("current_has_no_summary",
          not any(column.startswith("dmean__") or column.startswith("dstd__")
                  or column.startswith("dslope__")
                  for column in current_columns))
    check("summary_adds_only_summary_stats",
          set(current_columns).issubset(set(summary_columns))
          and len(set(summary_columns) - set(current_columns)) == 24)
    check("feature_revision_is_v2", FEATURE_CODE_VERSION.endswith("v2"))

    # 6. Inclinação por dia real.
    day = pd.Timestamp("2024-01-01")
    _, _, slope = _daily_stats(pd.Series([day, day + pd.Timedelta(days=1)]),
                               np.array([0.0, 2.0]))
    check("slope_per_actual_day", abs(slope - 2.0) < 1e-12)

    # 7. Splits disjuntos e cobertura completa.
    check("split_disjoint", all(
        set(row["test_ids"]).isdisjoint(set(row["outer_train_ids"]))
        and set(row["val_ids"]).isdisjoint(set(row["inner_train_ids"]))
        for row in SPLITS.to_dict(orient="records")
    ))

    # 8. Imputador/scaler ajustados só no treino, incluindo coluna toda NaN.
    x_train = pd.DataFrame({
        "cur__s0": [1.0, 2.0, np.nan],
        "cur__s1": [np.nan, np.nan, np.nan],
    })
    preprocessor = TabularPreprocessor("standard").fit(x_train)
    check("all_nan_fallback_constant",
          "cur__s1" in preprocessor.constant_fallback_columns_
          and preprocessor.fill_values_["cur__s1"] == 0.0)
    none_preprocessor = TabularPreprocessor("none").fit(x_train)
    forced = none_preprocessor.transform(pd.DataFrame({
        "cur__s0": [5.0], "cur__s1": [99.0]}))
    check("all_nan_column_stays_constant_on_new_data",
          forced[0, 1] == 0.0 and forced[0, 3] == 0.0)
    mean_before = preprocessor.scaler_.mean_.copy()
    transformed = preprocessor.transform(x_train)
    check("all_nan_transform_finite", np.isfinite(transformed).all())
    check("fixed_missing_indicators",
          len(preprocessor.indicator_columns_) == x_train.shape[1])
    changed = x_train.copy()
    changed.loc[0, "cur__s0"] = 1e9
    preprocessor.transform(changed)
    check("scaler_not_refit_on_transform",
          np.array_equal(mean_before, preprocessor.scaler_.mean_))
    check("signed_log_keeps_negative",
          signed_log1p(np.array([-3.0, 3.0]))[0] < 0)

    # 9. Pesos: componente e componente-fase, ambos média normalizada.
    ids = np.array([1, 1, 1, 2])
    phases = np.array(["saudavel", "pos_FPT", "pos_FPT", "saudavel"])
    component_weights = compute_weights(ids, phases, "component")
    check("component_weight_mean_one", abs(component_weights.mean() - 1.0) < 1e-12)
    sums = [component_weights[ids == component].sum()
            for component in sorted(set(ids))]
    check("component_weight_equal_components", abs(sums[0] - sums[1]) < 1e-12)
    phase_weights = compute_weights(ids, phases, "component_phase")
    check("component_phase_weight_mean_one", abs(phase_weights.mean() - 1.0) < 1e-12)
    component_sums = [phase_weights[ids == component].sum()
                      for component in sorted(set(ids))]
    check("component_phase_equal_components",
          abs(component_sums[0] - component_sums[1]) < 1e-12)
    for component in sorted(set(ids)):
        cell_sums = [
            phase_weights[(ids == component) & (phases == phase)].sum()
            for phase in sorted(set(phases[ids == component]))
        ]
        check(f"component_phase_cells_equal_{component}",
              max(cell_sums) - min(cell_sums) < 1e-12)

    # 10. Unidades nativas e ausência de clamp.
    day_native, day_metric = _cer_rul.scaled_prediction_to_units(
        np.array([0.5]), "dias-90", label_meta=_cer_rul.LABEL_META)
    fraction_native, fraction_metric = scaled_prediction_to_units(
        np.array([0.5]), "porcentagem-data")
    negative_native, _ = _cer_rul.scaled_prediction_to_units(
        np.array([-0.2]), "dias-90", label_meta=_cer_rul.LABEL_META)
    check("day_units", abs(day_native[0] - 50.0) < 1e-12
          and abs(day_metric[0] - 50.0) < 1e-12)
    check("fraction_units_pp", abs(fraction_native[0] - 0.5) < 1e-12
          and abs(fraction_metric[0] - 50.0) < 1e-12)
    check("no_clamp_negative", negative_native[0] < 0)
    plot_y, plot_pred, plot_unit = plot_metric_columns("porcentagem-data")
    check("plot_uses_metric_columns",
          plot_y == "y_metric" and plot_pred == "pred_metric"
          and "pontos percentuais" in plot_unit)
    plot_target = pd.DataFrame({
        "endpoint_id": ["F1"], "Falha": [1], "equipment": ["A"],
        "Date": [pd.Timestamp("2024-01-01")], "phase": ["saudavel"],
        "RUL": [0.5], "y_metric": [50.0],
    })
    plot_frame = predictions_frame(
        plot_target, np.array([0.55]), "porcentagem-data",
        {"fold_id": "fold_A", "test_equipment": "A"},
        {"candidate_id": "plot", "family": "mean"},
    )
    check("plot_values_are_pp",
          abs(plot_frame["y_metric"].iloc[0] - 50.0) < 1e-12
          and abs(plot_frame["pred_metric"].iloc[0] - 55.0) < 1e-12)

    # 10b. Macro entre folds não mistura famílias.
    macro_input = pd.DataFrame([
        {"label": LABEL_ORDER[0], "family": "mean",
          "unit": PROTOCOL["labels"][LABEL_ORDER[0]]["unit"],
         "fold_id": "fold_A", "test_equipment": "A", "endpoint_id": "m1",
         "Falha": 1, "equipment": "A", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 12.0},
        {"label": LABEL_ORDER[0], "family": "mean",
          "unit": PROTOCOL["labels"][LABEL_ORDER[0]]["unit"],
         "fold_id": "fold_B", "test_equipment": "B", "endpoint_id": "m2",
         "Falha": 2, "equipment": "B", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 8.0},
        {"label": LABEL_ORDER[0], "family": "ridge",
          "unit": PROTOCOL["labels"][LABEL_ORDER[0]]["unit"],
         "fold_id": "fold_A", "test_equipment": "A", "endpoint_id": "r1",
         "Falha": 1, "equipment": "A", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 14.0},
        {"label": LABEL_ORDER[0], "family": "ridge",
          "unit": PROTOCOL["labels"][LABEL_ORDER[0]]["unit"],
         "fold_id": "fold_B", "test_equipment": "B", "endpoint_id": "r2",
         "Falha": 2, "equipment": "B", "phase": "saudavel",
         "y_metric": 10.0, "pred_metric": 6.0},
    ])
    macro = aggregate_metric_rows(fold_metric_rows(macro_input))
    mean_macro = macro[(macro["family"] == "mean")
                       & (macro["scope"] == "overall")
                       & (macro["name"] == "all")]["mae"].iloc[0]
    ridge_macro = macro[(macro["family"] == "ridge")
                        & (macro["scope"] == "overall")
                        & (macro["name"] == "all")]["mae"].iloc[0]
    check("macro_keeps_families_separate",
          abs(mean_macro - 2.0) < 1e-12 and abs(ridge_macro - 4.0) < 1e-12)

    # 10c. Completude esperada de payloads por família.
    expected_payload_count = len(LABEL_ORDER) * len(SPLITS) * len(MODEL_FAMILIES)
    check("expected_payload_count", expected_run_count() == expected_payload_count)
    payload_paths = {
        run_file_path(label, fold["fold_id"], family)
        for label in LABEL_ORDER
        for fold in SPLITS.to_dict(orient="records")
        for family in MODEL_FAMILIES
    }
    check("payload_paths_include_family", len(payload_paths) == expected_payload_count)
    check("eda_sensor_rows", len(SENSOR_EDA) == len(SENSOR_COLUMNS))
    check("eda_gap_rows", len(TIME_GAP_EDA) == len(FAULT_IDS))

    # 11. Fingerprint e retomada atômica.
    check("fingerprint_order_independent",
          sha256_json({"a": 1, "b": 2}) == sha256_json({"b": 2, "a": 1}))
    check("fingerprint_changes",
          sha256_json({"a": 1}) != sha256_json({"a": 2}))
    with tempfile.TemporaryDirectory(dir=str(CACHE_DIR)) as temporary:
        temporary = Path(temporary)
        path = temporary / "run.json"
        payload = {
            "run_identity": "id-teste",
            "label": "dias-90",
            "family": "mean",
            "fold_id": "fold_A",
            "candidate_mode": "quick",
            "chosen_candidate": {"candidate_id": "abc"},
            "metrics": {"overall": {"mae": 1.0}},
            "counts": {"n_outer_test": 1},
            "predictions": [{"endpoint_id": "F01-r0005"}],
        }
        check("resume_fixture_matches_required_schema",
              set(payload) == set(RUN_REQUIRED_KEYS))
        save_complete_run(path, payload)
        loaded = load_complete_run(path, "id-teste", ["F01-r0005"])
        check("resume_complete_load", loaded is not None
              and len(loaded["predictions"]) == 1)
        broken = dict(payload)
        broken.pop("predictions")
        atomic_write_json(path, seal_payload(broken))
        try:
            load_complete_run(path, "id-teste", ["F01-r0005"])
            raise AssertionError("JSON sem predições deveria ser rejeitado")
        except RunIntegrityError:
            results.append("resume_missing_predictions_rejected")
        save_complete_run(path, payload)
        tampered = read_json(path)
        tampered["metrics"]["overall"]["mae"] = 999.0
        atomic_write_json(path, tampered)
        try:
            load_complete_run(path, "id-teste", ["F01-r0005"])
            raise AssertionError("JSON adulterado deveria falhar na integridade")
        except RunIntegrityError:
            results.append("resume_bad_integrity_rejected")

    # 12. Nenhuma feature com nomes proibidos.
    for width in PROTOCOL["bin_hours_candidates"]:
        for representation in REPRESENTATIONS:
            for column in feature_columns(width, representation, True):
                if any(token in column for token in FORBIDDEN_INFERENCE_TOKENS):
                    raise AssertionError(f"feature proibida: {column}")
    results.append("no_forbidden_feature_names")

    print(f"OK — {len(results)} verificações leves passaram.")
    return results


SELF_TEST_RESULTS = run_self_tests()


In [ ]:
# =============================================================================
# SEÇÃO 14 — AGREGAÇÃO, MANIFESTO, COMPARAÇÃO OPCIONAL E ZIP
# =============================================================================


def write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def export_data_tables():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    write_csv(AUDIT_TABLE, RUN_DIR / "cleanup_audit.csv")
    atomic_write_json(RUN_DIR / "cleanup_audit_full.json", AUDITS)
    write_csv(SENSOR_EDA, RUN_DIR / "eda_sensor_quantiles.csv")
    write_csv(TIME_GAP_EDA, RUN_DIR / "eda_time_gaps.csv")
    write_csv(TERMINAL_TABLE, RUN_DIR / "terminal_targets.csv")
    endpoint_export = ENDPOINTS.copy()
    endpoint_export["Date"] = endpoint_export["Date"].map(
        lambda value: pd.Timestamp(value).isoformat())
    write_csv(endpoint_export, RUN_DIR / "endpoint_index.csv")
    split_export = SPLITS.copy()
    for column in ("test_ids", "val_ids", "inner_train_ids", "outer_train_ids"):
        split_export[column] = split_export[column].map(
            lambda values: ",".join(str(value) for value in values))
    for column in ("inner_train_equipment",):
        split_export[column] = split_export[column].map(
            lambda values: ",".join(str(value) for value in values))
    write_csv(split_export, RUN_DIR / "splits.csv")
    atomic_write_json(RUN_DIR / "data_identity.json", DATA_IDENTITY)


def predictions_from_payloads(payloads):
    rows = []
    for payload in payloads:
        rows.extend(payload["predictions"])
    if not rows:
        return pd.DataFrame(columns=[
            "label", "unit", "fold_id", "test_equipment", "endpoint_id",
            "Falha", "equipment", "Date", "phase", "candidate_id", "family",
            "y_native", "pred_native", "y_metric", "pred_metric",
            "residual_metric", "split_role",
        ])
    frame = pd.DataFrame(rows)
    frame["Date"] = pd.to_datetime(frame["Date"])
    return frame


def chosen_config_rows(payloads):
    rows = []
    for payload in payloads:
        candidate = payload["chosen_candidate"]
        rows.append({
            "label": payload["label"],
            "fold_id": payload["fold_id"],
            "test_equipment": payload["fold"]["test_equipment"],
            "candidate_mode": payload["candidate_mode"],
            "candidate_id": candidate["candidate_id"],
            "family": payload["family"],
            "head": payload["head"],
            "horizon_days": candidate["horizon_days"],
            "bin_hours": candidate["bin_hours"],
            "representation": candidate["representation"],
            "preprocessing": candidate["preprocessing"],
            "add_missing_indicators": candidate["add_missing_indicators"],
            "timing": candidate["timing"],
            "weights": candidate["weights"],
            "seed": candidate["seed"],
            "params": canonical_json(candidate["params"]),
            "selection_score": payload["chosen_record"]["score"],
            "inner_component_mae": canonical_json(
                payload["chosen_record"].get("component_mae")),
            "forecast_cache_key": payload.get("forecast_cache_key"),
        })
    return pd.DataFrame(rows)


def trials_rows(payloads):
    if not isinstance(payloads, (list, tuple)):
        raise TypeError(
            "trials_rows espera uma LISTA de payloads de execução, não "
            f"{type(payloads).__name__}."
        )
    rows = []
    for payload in payloads:
        if not isinstance(payload, dict):
            raise TypeError(
                "Cada item de trials_rows deve ser um payload dict, não "
                f"{type(payload).__name__}."
            )
        rows.extend(payload.get("search_trials", []))
    if not rows:
        return pd.DataFrame()
    result = pd.DataFrame(rows)
    result["candidate"] = result["candidate"].map(canonical_json)
    result["component_mae"] = result["component_mae"].map(
        lambda value: canonical_json(value) if value is not None else "")
    return result


def plot_predictions(prediction_frame):
    if prediction_frame.empty:
        return []
    outputs = []
    for (label, family), label_frame in prediction_frame.groupby(
            ["label", "family"], sort=False):
        folds = sorted(label_frame["fold_id"].unique())
        metric_y, metric_pred, unit = plot_metric_columns(label)
        figure, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=False)
        axes = np.ravel(axes)
        for axis, fold_id in zip(axes, folds):
            fold_frame = label_frame[label_frame["fold_id"] == fold_id]
            for fault_id, component in fold_frame.groupby("Falha", sort=True):
                component = component.sort_values("Date")
                axis.plot(component["Date"], component[metric_y],
                          color="black", linewidth=1.0, alpha=0.55)
                axis.plot(component["Date"], component[metric_pred],
                          linewidth=1.2, label=f"Falha {int(fault_id)}")
            axis.set_title(f"{label} | {family} | {fold_id} | teste "
                           f"{fold_frame['test_equipment'].iloc[0]}")
            axis.set_ylabel(unit)
            axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            axis.tick_params(axis="x", rotation=30)
            axis.grid(alpha=0.25)
            axis.legend(fontsize=7, ncol=2)
        for axis in axes[len(folds):]:
            axis.axis("off")
        figure.suptitle(f"Predições em datas reais — {label} / {family} (sem clamp)")
        figure.tight_layout()
        path = PLOTS_DIR / f"predicoes_{label}__{family}.png"
        figure.savefig(path, dpi=140)
        plt.close(figure)
        outputs.append(path)
    return outputs


def artifact_hashes(root):
    artifacts = {}
    for path in sorted(Path(root).rglob("*")):
        if not path.is_file() or path.name == "manifest.json":
            continue
        artifacts[path.relative_to(root).as_posix()] = sha256_file(path)
    return artifacts


BASELINE_FAMILIES = ("mean", "ridge", "svr", "extra_trees")
COMPARISON_REQUIRED_COLUMNS = (
    "label", "family", "fold_id", "test_equipment", "endpoint_id", "Falha",
    "equipment", "phase", "unit", "y_native", "y_metric", "pred_native",
    "pred_metric", "split_role",
)


def expected_comparison_records():
    """Mapa completo e independente dos grupos baseline esperados."""
    expected = {}
    for label in LABEL_ORDER:
        meta = PROTOCOL["labels"][label]
        for family in BASELINE_FAMILIES:
            for fold in SPLITS.to_dict(orient="records"):
                masks = masks_for_fold(TARGETS[label], fold)
                subset = TARGETS[label].loc[masks["test"]]
                for row in subset.itertuples(index=False):
                    key = (label, family, fold["fold_id"], row.endpoint_id)
                    if key in expected:
                        raise RuntimeError(f"Grupo de comparação duplicado: {key}")
                    expected[key] = {
                        "Falha": int(row.Falha),
                        "equipment": str(row.equipment),
                        "test_equipment": fold["test_equipment"],
                        "phase": str(row.phase),
                        "unit": meta["unit"],
                        "y_native": float(row.RUL),
                        "y_metric": float(row.y_metric),
                        "split_role": "outer_test",
                    }
    return expected


def validate_comparison_predictions(frame, expected):
    """Valida cobertura, duplicatas, equipamento, fase, unidade e ground truth."""
    missing_columns = [column for column in COMPARISON_REQUIRED_COLUMNS
                       if column not in frame.columns]
    if missing_columns:
        raise RuntimeError(
            f"Predições baseline sem colunas: {missing_columns}"
        )
    seen = set()
    for row in frame.itertuples(index=False):
        key = (str(row.label), str(row.family), str(row.fold_id),
               str(row.endpoint_id))
        if key in seen:
            raise RuntimeError(f"Predição baseline duplicada: {key}")
        if key not in expected:
            raise RuntimeError(f"Predição baseline extra/inesperada: {key}")
        expected_row = expected[key]
        if int(row.Falha) != expected_row["Falha"]:
            raise RuntimeError(f"Falha incompatível: {key}")
        if str(row.equipment) != expected_row["equipment"]:
            raise RuntimeError(f"Equipamento incompatível: {key}")
        if str(row.test_equipment) != expected_row["test_equipment"]:
            raise RuntimeError(f"Equipamento de teste incompatível: {key}")
        if str(row.phase) != expected_row["phase"]:
            raise RuntimeError(f"Fase incompatível: {key}")
        if str(row.unit) != expected_row["unit"]:
            raise RuntimeError(f"Unidade incompatível: {key}")
        if str(row.split_role) != expected_row["split_role"]:
            raise RuntimeError(f"Split role incompatível: {key}")
        if not np.isclose(float(row.y_native), expected_row["y_native"],
                          rtol=0.0, atol=1e-9):
            raise RuntimeError(f"Ground truth y_native incompatível: {key}")
        if not np.isclose(float(row.y_metric), expected_row["y_metric"],
                          rtol=0.0, atol=1e-9):
            raise RuntimeError(f"Ground truth y_metric incompatível: {key}")
        if not np.isfinite(float(row.pred_native)) or not np.isfinite(float(row.pred_metric)):
            raise RuntimeError(f"Predição baseline não finita: {key}")
        seen.add(key)
    missing = set(expected) - seen
    if missing:
        preview = sorted(missing)[:3]
        raise RuntimeError(
            f"{len(missing)} predições baseline ausentes; exemplos={preview}"
        )
    return frame


def load_baseline_comparison(current_macro):
    cfg = CONFIG["comparison"]
    if not cfg["enabled"]:
        return {"enabled": False}
    if not CONFIG["run_training"] or current_macro is None or current_macro.empty:
        return {
            "enabled": True,
            "status": "deferred_no_current_metrics",
            "baseline_run_dir": str(cfg["baseline_run_dir"]),
            "reason": ("a comparação só é materializada quando há métricas "
                       "Timestral produzidas por treino completo; o smoke de "
                       "dados não lê nem usa scores importados."),
            "selection_uses_imported_test_score": False,
        }
    baseline_dir = Path(cfg["baseline_run_dir"])
    manifest_path = baseline_dir / "manifest.json"
    predictions_path = baseline_dir / "predictions_all.csv"
    macro_path = baseline_dir / "metrics_macro.csv"
    if not manifest_path.is_file() or not predictions_path.is_file():
        raise FileNotFoundError(
            "comparison.baseline_run_dir precisa conter manifest.json e "
            "predictions_all.csv exportados pelo notebook de baselines."
        )
    baseline_manifest = read_json(manifest_path)
    if baseline_manifest.get("status") != "training_complete":
        raise RuntimeError("Comparação exige baseline com status training_complete.")
    checks = {
        "data_identity_hash": (baseline_manifest.get("data_identity_hash"),
                               DATA_IDENTITY_HASH),
        "dataset_id": ((baseline_manifest.get("protocol") or {}).get("dataset_id"),
                       DATASET_ID),
        "protocol_hash": (baseline_manifest.get("protocol_hash"), PROTOCOL_HASH),
        "endpoint_sha256": (baseline_manifest.get("endpoints", {}).get("sha256"),
                            ENDPOINT_HASH),
        "split_sha256": (baseline_manifest.get("splits", {}).get("sha256"),
                         SPLIT_HASH),
    }
    mismatches = {key: values for key, values in checks.items()
                  if values[0] != values[1]}
    if mismatches:
        raise RuntimeError(f"Comparação recusada; identidades divergentes: {mismatches}")
    training = baseline_manifest.get("training", {})
    expected_baseline_runs = len(LABEL_ORDER) * len(SPLITS) * len(BASELINE_FAMILIES)
    expected_per_family = len(LABEL_ORDER) * len(SPLITS)
    if int(training.get("expected_runs", -1)) != expected_baseline_runs:
        raise RuntimeError(
            f"Manifest baseline não declara {expected_baseline_runs} execuções esperadas.")
    if int(training.get("observed_runs", -1)) != expected_baseline_runs:
        raise RuntimeError(
            f"Manifest baseline não declara {expected_baseline_runs} execuções observadas.")
    counts_by_family = training.get("counts_by_family", {})
    if set(counts_by_family) != set(BASELINE_FAMILIES) or any(
            int(counts_by_family[family]) != expected_per_family
            for family in BASELINE_FAMILIES):
        raise RuntimeError("Contagens por família baseline incompatíveis.")

    artifacts = baseline_manifest.get("artifacts", {})
    for filename, path in (("predictions_all.csv", predictions_path),
                           ("metrics_macro.csv", macro_path)):
        if filename not in artifacts:
            raise RuntimeError(f"Manifest baseline sem hash de {filename}.")
        if artifacts[filename] != sha256_file(path):
            raise RuntimeError(f"Hash do artefato baseline divergente: {filename}")

    baseline_predictions = pd.read_csv(predictions_path)
    expected = expected_comparison_records()
    baseline_predictions = validate_comparison_predictions(
        baseline_predictions, expected
    )
    baseline_macro = aggregate_metric_rows(fold_metric_rows(baseline_predictions))
    if baseline_macro.empty:
        raise RuntimeError("Macro baseline recomputada vazia.")
    baseline_table = baseline_macro.copy()
    baseline_table["source_pipeline"] = "baseline_standalone"
    baseline_table["pipeline_role"] = "baseline"
    current_table = current_macro.copy()
    if current_table.empty:
        raise RuntimeError("Macro Timestral vazia para comparação.")
    current_table["source_pipeline"] = "timestral_hybrid"
    current_table["pipeline_role"] = "hybrid"
    comparison_table = pd.concat([baseline_table, current_table],
                                 ignore_index=True, sort=False)
    write_csv(comparison_table, RUN_DIR / "comparison_metrics.csv")
    return {
        "enabled": True,
        "baseline_run_dir": str(baseline_dir),
        "identity_checks": {key: {"baseline": values[0], "current": values[1]}
                            for key, values in checks.items()},
        "artifact_hash_checks": {
            filename: {"baseline": artifacts[filename],
                       "recomputed": sha256_file(path)}
            for filename, path in (("predictions_all.csv", predictions_path),
                                   ("metrics_macro.csv", macro_path))
        },
        "expected_groups": int(len(expected)),
        "validated_predictions_rows": int(len(baseline_predictions)),
        "units_checked": True,
        "ground_truth_checked": True,
        "baseline_macro": baseline_macro.to_dict(orient="records"),
        "comparison_table_file": "comparison_metrics.csv",
        "comparison_table": comparison_table.to_dict(orient="records"),
        "note": ("comparação descritiva de pipelines independentemente ajustadas; "
                 "não isola o efeito do Timestral e não seleciona modelo"),
        "selection_uses_imported_test_score": False,
    }


def build_manifest(payloads, preview, fit_counter, started_utc, finished_utc,
                   metric_macro, ablation_results, forecast_preview):
    chosen = chosen_config_rows(payloads)
    trial_table = trials_rows(payloads)
    comparison = load_baseline_comparison(metric_macro)
    observed = int(len(payloads))
    expected = expected_run_count()
    if CONFIG["run_training"] and observed == expected:
        status = "training_complete"
    elif not CONFIG["run_training"] and not CONFIG["model_specific"]["forecast_smoke"]:
        status = "data_smoke_without_training_or_forecast"
    elif not CONFIG["run_training"] and CONFIG["model_specific"]["forecast_smoke"]:
        status = "forecast_smoke_only"
    else:
        status = "incomplete"
    fit_rows = []
    for payload in payloads:
        fit_rows.append({
            "label": payload["label"],
            "family": payload["family"],
            "head": payload["head"],
            "fold_id": payload["fold_id"],
            "model_fits": payload["fit_counts_this_run"].get("model_fits", 0),
            "selection_fits": payload["fit_counts_this_run"].get("selection_fits", 0),
            "final_fits": payload["fit_counts_this_run"].get("final_fits", 0),
        })
    manifest = {
        "status": status,
        "pipeline_id": TIMESTRAL["pipeline_id"],
        "run_id": RUN_ID,
        "started_utc": started_utc,
        "finished_utc": finished_utc,
        "source": SOURCE_PROVENANCE,
        "dataset": {
            "dataset_id": DATASET_ID,
            "unit_kind": UNIT_KIND,
            "unit_label": UNIT_LABEL_PLURAL,
            "group_count": len(EQUIPMENT_ORDER),
            "groups": EQUIPMENT_ORDER,
            "labels": LABEL_ORDER,
        },
        "preprocessing": PREPROCESSING_CONFIG,
        "protocol": PROTOCOL,
        "versions": {**RUNTIME_VERSIONS, "timestral": TIMESTRAL_VERSIONS},
        "config": CONFIG,
        "effective_config": EFFECTIVE_CONFIG,
        "effective_config_hash": EFFECTIVE_CONFIG_HASH,
        "protocol_hash": PROTOCOL_HASH,
        "data_identity": DATA_IDENTITY,
        "data_identity_hash": DATA_IDENTITY_HASH,
        "data": {
            "input_files": [
                {"label": label, "filename": INPUT_FILES[label].name,
                 "sha256": RAW_FILE_HASHES[label]}
                for label in LABEL_ORDER
            ],
            "clean_frame_sha256": CLEAN_HASHES,
            "cleanup_audit": AUDIT_TABLE.to_dict(orient="records"),
            "cleanup_audit_full": "cleanup_audit_full.json",
            "eda_sensor_quantiles": "eda_sensor_quantiles.csv",
            "eda_time_gaps": "eda_time_gaps.csv",
            "terminal_targets": TERMINAL_TABLE.assign(
                Date=TERMINAL_TABLE["Date"].map(
                    lambda value: pd.Timestamp(value).isoformat())
            ).to_dict(orient="records"),
            "counts": {
                label: {
                    "raw_rows": AUDITS[label]["raw_rows"],
                    "clean_keys": AUDITS[label]["clean_keys"],
                    "target_conflict_keys": AUDITS[label]["target_conflict_keys"],
                    "terminal_nonzero_count": AUDITS[label]["terminal_nonzero_count"],
                }
                for label in LABEL_ORDER
            },
        },
        "endpoints": {
            "sha256": ENDPOINT_HASH,
            "count": int(len(ENDPOINTS)),
            "per_fault": ENDPOINT_COUNTS,
            "warmup_records": PROTOCOL["warmup_records"],
            "stride": PROTOCOL["endpoint_stride"],
            "coverage": "post_warmup_only_not_all_original_rows",
        },
        "splits": {
            "sha256": SPLIT_HASH,
            "folds": SPLITS.to_dict(orient="records"),
        },
        "forecast": {
            "model_specific": TIMESTRAL,
            "versions": TIMESTRAL_VERSIONS,
            "source_audit": TIMESTRAL_SOURCE_AUDIT,
            "plan": forecast_preview["unique_configs"],
            "preview": forecast_preview,
            "actual_counts": dict(FORECAST_COUNTER),
            "cache_metadata": TIMESTRAL_CACHE_METADATA,
            "identity_contains_targets": False,
            "no_forecast_control_trained_here": True,
            "sensor_uncertainty_is_not_rul_interval": True,
            "mock_forecasts_only_in_self_tests": True,
        },
        "training": {
            "run_training": bool(CONFIG["run_training"]),
            "run_forecast": bool(TIMESTRAL["run_forecast"]),
            "forecast_smoke": bool(TIMESTRAL["forecast_smoke"]),
            "expected_runs": expected,
            "observed_runs": observed,
            "counts_by_family": {
                family: int(sum(payload["family"] == family for payload in payloads))
                for family in MODEL_FAMILIES
            },
            "preview_fit_counts": preview,
            "actual_fit_counts": dict(fit_counter),
            "fit_counts_by_run": fit_rows,
            "chosen_configs": chosen.to_dict(orient="records"),
            "search_trials": trial_table.to_dict(orient="records"),
            "metric_macro": metric_macro.to_dict(orient="records")
            if isinstance(metric_macro, pd.DataFrame) and not metric_macro.empty else [],
            "ablation": ablation_results,
            "notes": [
                "cada head escolhe sua configuração apenas pela MAE macro por componente no fold interno",
                "Ridge e ExtraTrees são refeitos e testados separadamente; não há vencedor global",
                "não há média constante, SVR ou repetição de baselines standalone",
                "forecasts congelados de sensores são reutilizados entre rótulos/folds; a RUL nunca entra no cache",
                "não há controle sem forecast nesta notebook; a comparação opcional é descritiva",
                "sem clamp, MAPE ou escala do ciclo de vida do teste",
            ],
        },
        "comparison": comparison,
        "artifacts": {},
    }
    return manifest


def export_results(payloads, preview, fit_counter, started_utc, finished_utc,
                   ablation_results=None, forecast_preview=None):
    if ablation_results is None:
        ablation_results = {}
    if forecast_preview is None:
        forecast_preview = forecast_cost_preview()
    export_data_tables()
    predictions = predictions_from_payloads(payloads)
    metric_rows = fold_metric_rows(predictions) if not predictions.empty else []
    metrics_folds = pd.DataFrame(metric_rows)
    metrics_macro = aggregate_metric_rows(metric_rows)
    if not metrics_folds.empty:
        write_csv(metrics_folds, RUN_DIR / "metrics_folds.csv")
    if not metrics_macro.empty:
        write_csv(metrics_macro, RUN_DIR / "metrics_macro.csv")
        write_csv(metrics_macro[metrics_macro["scope"] == "equipment_macro"],
                  RUN_DIR / "equipment_macro.csv")
    write_csv(predictions, RUN_DIR / "predictions_all.csv")
    chosen = chosen_config_rows(payloads)
    write_csv(chosen, RUN_DIR / "chosen_configs.csv")
    trial_table = trials_rows(payloads)
    if not trial_table.empty:
        write_csv(trial_table, RUN_DIR / "search_trials.csv")
    fit_rows = []
    for payload in payloads:
        fit_rows.append({
            "label": payload["label"],
            "family": payload["family"],
            "head": payload["head"],
            "fold_id": payload["fold_id"],
            "candidate_mode": payload["candidate_mode"],
            "candidate_id": payload["chosen_candidate"]["candidate_id"],
            "horizon_days": payload["chosen_candidate"]["horizon_days"],
            "model_fits": payload["fit_counts_this_run"].get("model_fits", 0),
            "selection_fits": payload["fit_counts_this_run"].get("selection_fits", 0),
            "final_fits": payload["fit_counts_this_run"].get("final_fits", 0),
            "forecast_batches": payload["forecast_counts_this_run"].get("batches", 0),
            "forecast_endpoints": payload["forecast_counts_this_run"].get("endpoints", 0),
        })
    write_csv(pd.DataFrame(fit_rows), RUN_DIR / "fit_counts_by_run.csv")
    write_csv(pd.DataFrame([{
        "batches": int(FORECAST_COUNTER.get("batches", 0)),
        "endpoints": int(FORECAST_COUNTER.get("endpoints", 0)),
        "run_forecast": bool(TIMESTRAL["run_forecast"]),
        "forecast_smoke": bool(TIMESTRAL["forecast_smoke"]),
    }]), RUN_DIR / "forecast_counts.csv")
    atomic_write_json(RUN_DIR / "forecast_plan.json",
                      {"plan": [list(item) for item in forecast_plan()],
                       "preview": forecast_preview})
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    plot_paths = plot_predictions(predictions)
    manifest = build_manifest(payloads, preview, fit_counter, started_utc,
                              finished_utc, metrics_macro, ablation_results,
                              forecast_preview)
    manifest["artifacts"] = artifact_hashes(RUN_DIR)
    atomic_write_json(RUN_DIR / "manifest.json", manifest)

    zip_path = EXPORTS_DIR / f"{RUN_ID}.zip"
    temporary = zip_path.with_name(f".{zip_path.name}.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=(Path(RUN_ID) /
                                             path.relative_to(RUN_DIR)).as_posix())
    os.replace(temporary, zip_path)
    print("ZIP de resultados:", zip_path)
    print("CACHE, pesos e arquivos de entrada NÃO foram incluídos no ZIP.")
    if CONFIG["output"]["download_zip"] and IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
    return {
        "predictions": predictions,
        "metrics_folds": metrics_folds,
        "metrics_macro": metrics_macro,
        "chosen": chosen,
        "trials": trial_table,
        "manifest": manifest,
        "zip_path": zip_path,
        "plots": plot_paths,
    }


print("Relatórios Timestral prontos.")


## Testes leves

- As células `common-tests` são as mesmas do notebook de baselines e verificam limpeza, endpoints, grade, imputador, pesos, unidades, persistência e contagem de payloads derivada de `MODEL_FAMILIES` (80 baselines; 40 Timestral).
- A célula `timestral-tests` usa **forecasts mock apenas em teste**: valida conversão de horizonte, atributos `q50/q10/q90`, causalidade do contexto, rejeição de shape/NaN e uma chave de cache que não contém rótulo/alvo.
- Nenhum mock é usado na execução científica. `run_training=True` exige `run_forecast=True` e chama o Timestral real; `forecast_smoke=True` é o único caminho de inferência real sem treinar heads.
- A execução científica no Colab permanece UNVERIFIED localmente. O smoke de dados não importa torch/Timestral.


In [ ]:
# =============================================================================
# SEÇÃO 16 — TESTES TIMESTRAL LEVES (MOCK; SEM INFERÊNCIA REAL)
# =============================================================================


def _mock_forecast_arrays():
    n, sensors, steps = 2, len(SENSOR_KEYS), horizon_steps(3, 12)
    base = np.arange(steps, dtype=float)[None, None, :]
    median = np.broadcast_to(base, (n, sensors, steps)).copy()
    q10 = median - 1.0
    q90 = median + 2.0
    current = np.ones((n, sensors), dtype=float)
    return median, q10, q90, current


def run_timestral_self_tests():
    results = []

    def check(name, condition):
        if not condition:
            raise AssertionError(name)
        results.append(name)

    check("horizon_1d_12h", horizon_steps(1, 12) == 2)
    check("horizon_3d_12h", horizon_steps(3, 12) == 6)
    check("horizon_7d_12h", horizon_steps(7, 12) == 14)
    check("horizon_3d_6h", horizon_steps(3, 6) == 12)
    check("horizon_3d_24h", horizon_steps(3, 24) == 3)
    try:
        horizon_steps(0.5, 24)
        raise AssertionError("horizonte fracionário deveria falhar")
    except ValueError:
        results.append("horizon_conversion_rejects_fraction")

    check("forecast_40_features",
          len(TIMESTRAL_FORECAST_COLUMNS) == 5 * len(SENSOR_KEYS))
    check("forecast_no_raw_trajectory_names",
          not any("future" in column or "trajectory" in column
                  for column in TIMESTRAL_FORECAST_COLUMNS))
    check("forecast_no_forbidden_names",
          not any(any(token in column for token in FORBIDDEN_INFERENCE_TOKENS)
                  for column in TIMESTRAL_FORECAST_COLUMNS))
    check("pipelines_are_timestral_prefixed",
          all(family.startswith("timestral_") for family in MODEL_FAMILIES))
    check("two_heads_only", set(MODEL_FAMILIES) == set(PIPELINE_BY_HEAD.values()))
    check("expected_payloads_derived",
          expected_run_count() == len(LABEL_ORDER) * len(SPLITS) * len(MODEL_FAMILIES))
    check("timestral_has_two_heads", len(MODEL_FAMILIES) == 2)

    median, q10, q90, current = _mock_forecast_arrays()
    frame = forecast_feature_frame(
        ["F1", "F2"], median, q10, q90, current, 12, 3)
    check("forecast_feature_shape", frame.shape == (2, 5 * len(SENSOR_KEYS)))
    check("forecast_feature_index", frame.index.tolist() == ["F1", "F2"])
    check("forecast_last_median", abs(frame.loc["F1", "fc_lastmedian__s0"] - 5.0) < 1e-12)
    check("forecast_mean_median", abs(frame.loc["F1", "fc_meanmedian__s0"] - 2.5) < 1e-12)
    check("forecast_delta_current", abs(frame.loc["F1", "fc_delta__s0"] - 4.0) < 1e-12)
    check("forecast_slope_per_day", abs(frame.loc["F1", "fc_slope_day__s0"] - 2.0) < 1e-12)
    check("forecast_width_mean", abs(frame.loc["F1", "fc_width__s0"] - 3.0) < 1e-12)
    check("forecast_feature_finite",
          np.isfinite(frame.loc[:, list(TIMESTRAL_FORECAST_COLUMNS)].to_numpy(float)).all())

    try:
        validate_forecast_arrays(median[:, :, :-1], q10, q90, len(median), 6)
        raise AssertionError("shape inválido deveria falhar")
    except ValueError:
        results.append("forecast_shape_validation_rejects")
    bad = median.copy()
    bad[0, 0, 0] = np.nan
    try:
        validate_forecast_arrays(bad, q10, q90, len(median), 6)
        raise AssertionError("NaN deveria falhar")
    except ValueError:
        results.append("forecast_finite_validation_rejects")
    bad_low = median + 10.0
    bad_high = median - 10.0
    try:
        validate_forecast_arrays(median, bad_low, bad_high, len(median), 6)
        raise AssertionError("ordenação de quantis inválida deveria falhar")
    except ValueError:
        results.append("forecast_quantile_order_rejects")
    bad_features = frame.copy()
    bad_features.iloc[0, 0] = np.nan
    try:
        validate_forecast_feature_frame(bad_features)
        raise AssertionError("features não finitas deveriam falhar")
    except ValueError:
        results.append("forecast_nonfinite_features_reject")

    identity = timestral_cache_identity_payload(12, 3)
    forbidden = {"label", "target", "clean_frame_sha256", "data_base_hash", "RUL"}
    check("cache_identity_excludes_labels_targets", not (forbidden & set(identity)))
    check("cache_identity_has_sensor_and_endpoint",
          identity["sensor_sha256"] == SENSOR_HASH
          and identity["endpoint_sha256"] == ENDPOINT_HASH)
    check("cache_identity_stable", timestral_cache_key(12, 3) == timestral_cache_key(12, 3))
    check("cache_identity_width_changes", timestral_cache_key(12, 3) != timestral_cache_key(6, 3))
    check("cache_identity_horizon_changes", timestral_cache_key(12, 3) != timestral_cache_key(12, 7))
    original_revision = TIMESTRAL["model_revision"]
    try:
        TIMESTRAL["model_revision"] = original_revision + "-changed"
        changed_key = timestral_cache_key(12, 3)
    finally:
        TIMESTRAL["model_revision"] = original_revision
    check("cache_identity_weights_revision_changes", changed_key != timestral_cache_key(12, 3))
    original_clip = TIMESTRAL["clip_nonnegative"]
    try:
        TIMESTRAL["clip_nonnegative"] = not original_clip
        changed_key = timestral_cache_key(12, 3)
    finally:
        TIMESTRAL["clip_nonnegative"] = original_clip
    check("cache_identity_options_change", changed_key != timestral_cache_key(12, 3))
    cache_key = timestral_cache_key(12, 3)
    metadata = forecast_cache_metadata(12, 3, cache_key, ["F1", "F2"], "cpu")
    check("cache_metadata_complete_identity",
          metadata["identity"] == identity
          and metadata["forecast_feature_columns"] == list(TIMESTRAL_FORECAST_COLUMNS))
    changed_metadata = json.loads(json.dumps(metadata))
    changed_metadata["identity"]["horizon_days"] = 7.0
    try:
        assert_forecast_cache_metadata(
            changed_metadata, 12, 3, cache_key, ["F1", "F2"])
        raise AssertionError("metadata divergente deveria falhar")
    except RuntimeError:
        results.append("forecast_metadata_mismatch_rejects")

    commit = CONFIG["model_specific"]["package_commit"]
    matching_url = {
        "url": "https://github.com/mistralai/timestral.git",
        "vcs_info": {"commit_id": commit, "requested_revision": commit},
    }
    verified = audit_timestral_source(
        direct_url=matching_url, installed_version="0.1",
        torch_version="2.4.0", python_version=(3, 10, 0))
    check("timestral_source_verified_fixture",
          verified["source_status"] == "verified"
          and verified["verified_package_commit"] == commit
          and verified["torch_ge_2_4"] and verified["python_ge_3_10"])
    missing = audit_timestral_source(
        direct_url={}, installed_version=None,
        torch_version="2.4.0", python_version=(3, 10, 0))
    check("timestral_source_missing_fixture",
          missing["source_status"] in ("missing_package", "missing_vcs_commit"))
    mismatched_url = {
        "url": "https://github.com/mistralai/timestral.git",
        "vcs_info": {"commit_id": "0" * 40, "requested_revision": "0" * 40},
    }
    mismatched = audit_timestral_source(
        direct_url=mismatched_url, installed_version="0.1",
        torch_version="2.4.0", python_version=(3, 10, 0))
    check("timestral_source_mismatch_fixture",
          mismatched["source_status"] == "commit_mismatch")

    context_group = _synthetic_frame(n=80)
    endpoint = pd.DataFrame([{
        "endpoint_id": "F01-r0060",
        "Falha": 1,
        "Date": context_group.loc[60, "Date"],
        "record_index": 60,
    }])
    contexts = build_timestral_contexts(context_group, endpoint, 12)
    check("context_shape", contexts.shape == (1, len(SENSOR_KEYS), PROTOCOL["context_bins"]))
    check("context_has_observed_values", np.isfinite(contexts).any())
    future = context_group.copy()
    extra = future.iloc[[-1]].copy()
    extra["Date"] = future["Date"].max() + pd.Timedelta(days=10)
    extra["Falha"] = 1
    future = pd.concat([future, extra], ignore_index=True)
    contexts_future = build_timestral_contexts(future, endpoint, 12)
    check("context_does_not_use_future",
          np.allclose(contexts, contexts_future, equal_nan=True))

    plan = set(forecast_plan())
    if CONFIG["search"]["enabled"]:
        active_pairs = {
            (int(width), float(days))
            for width in CONFIG["search"]["bin_hours_levels"]
            for days in CONFIG["search"]["horizon_days_levels"]
        }
    else:
        active_pairs = {
            (int(preset["bin_hours"]),
             float(preset.get("horizon_days", TIMESTRAL["default_horizon_days"])))
            for presets in CONFIG["quick"]["presets"].values()
            for preset in presets
        }
    check("forecast_plan_covers_active_config", active_pairs.issubset(plan))
    preview = forecast_cost_preview()
    check("forecast_preview_endpoints", preview["n_endpoints"] == len(ENDPOINTS))
    check("forecast_preview_batches", preview["batches_per_config"] >= 1)

    # Busca: o retorno real precisa expor head/family (evita KeyError silencioso).
    fold = SPLITS.to_dict(orient="records")[0]
    masks = masks_for_fold(TARGETS[LABEL_ORDER[0]], fold)
    stub_candidate = dict(quick_head_candidates(fold["fold_id"], "ridge")[0])
    stub_candidate["origin"] = "search:test"
    stub_candidate["candidate_id"] = timestral_candidate_id(stub_candidate)
    stub_result = {
        "head": "ridge", "family": "timestral_ridge", "study_id": "stub",
        "budget_total": 1, "trials_before": 0, "trials_after": 1,
        "new_trials": 1, "best_trial": 0, "best_score": 1.0,
        "candidate": stub_candidate,
        "trials": [{"trial_number": 0, "component_mae": {"1": 1.0},
                    "candidate": stub_candidate}],
    }
    original_enabled = CONFIG["search"]["enabled"]
    original_run_search_study = globals()["run_search_study"]
    globals()["run_search_study"] = lambda *args, **kwargs: stub_result
    CONFIG["search"]["enabled"] = True
    try:
        selection = select_head_inner(
            LABEL_ORDER[0], fold, "ridge", TARGETS[LABEL_ORDER[0]], masks, None, {})
        check("search_selection_exposes_head",
              selection["search_results"][0]["head"] == "ridge")
        check("search_selection_exposes_family",
              selection["search_results"][0]["family"] == "timestral_ridge")
        check("search_selection_keeps_family",
              selection["chosen"]["family"] == "ridge"
              and selection["chosen"]["pipeline_id"] == "timestral_ridge")
    finally:
        CONFIG["search"]["enabled"] = original_enabled
        globals()["run_search_study"] = original_run_search_study

    # Exportação: candidate/component_mae estruturados e serializados uma vez.
    trial_frame = trials_rows([{"search_trials": [{
        "label": LABEL_ORDER[0], "family": "timestral_ridge",
        "head": "ridge", "fold_id": fold["fold_id"], "study_id": "stub",
        "trial_number": 0, "state": "COMPLETE", "value": 1.0,
        "candidate": stub_candidate, "component_mae": {"1": 1.0}, "seed": 1,
    }]}])
    check("trials_candidate_single_json_encoding",
          isinstance(json.loads(trial_frame.iloc[0]["candidate"]), dict))
    check("trials_component_single_json_encoding",
          isinstance(json.loads(trial_frame.iloc[0]["component_mae"]), dict))

    # Comparação mock: cobertura/duplicata/ground truth validados sem ajuste.
    expected_one = {
        (LABEL_ORDER[0], "mean", "fold_A", "F1"): {
            "Falha": 1, "equipment": "A", "test_equipment": "A",
            "phase": "saudavel", "unit": PROTOCOL["labels"][LABEL_ORDER[0]]["unit"], "y_native": 10.0,
            "y_metric": 10.0, "split_role": "outer_test",
        }
    }
    comparison_row = {
        "label": LABEL_ORDER[0], "family": "mean", "fold_id": "fold_A",
        "test_equipment": "A", "endpoint_id": "F1", "Falha": 1,
        "equipment": "A", "phase": "saudavel", "unit": PROTOCOL["labels"][LABEL_ORDER[0]]["unit"],
        "y_native": 10.0, "y_metric": 10.0, "pred_native": 9.0,
        "pred_metric": 9.0, "split_role": "outer_test",
    }
    comparison_frame = pd.DataFrame([comparison_row])
    validate_comparison_predictions(comparison_frame, expected_one)
    results.append("comparison_predictions_valid")
    duplicated = pd.concat([comparison_frame, comparison_frame], ignore_index=True)
    try:
        validate_comparison_predictions(duplicated, expected_one)
        raise AssertionError("duplicata de comparação deveria falhar")
    except RuntimeError:
        results.append("comparison_duplicate_rejects")
    wrong_truth = comparison_frame.copy()
    wrong_truth.loc[0, "y_metric"] = 99.0
    try:
        validate_comparison_predictions(wrong_truth, expected_one)
        raise AssertionError("ground truth divergente deveria falhar")
    except RuntimeError:
        results.append("comparison_ground_truth_rejects")

    original_comparison = CONFIG["comparison"]["enabled"]
    CONFIG["comparison"]["enabled"] = True
    try:
        deferred = load_baseline_comparison(pd.DataFrame())
        check("comparison_deferred_without_training_metrics",
              deferred["enabled"] is True
              and deferred["status"] == "deferred_no_current_metrics"
              and deferred["selection_uses_imported_test_score"] is False)
    finally:
        CONFIG["comparison"]["enabled"] = original_comparison

    print(f"OK — {len(results)} verificações Timestral leves passaram (mock).")
    return results


TIMESTRAL_SELF_TEST_RESULTS = run_timestral_self_tests()


## Ablação opcional

A ablação é controlada e desligada por padrão. Ela mantém fixos parâmetros, horizonte, grade e seed, e altera **um único eixo** (`representation`, `timing` ou `weights`). Todos os níveis pré-declarados são refeitos e avaliados no teste externo; os resultados são exportados sem escolher um vencedor pelo teste.


In [ ]:
# =============================================================================
# SEÇÃO 15 — ABLAÇÃO OPCIONAL DE UM EIXO, SEM VENCEDOR EXTERNO
# =============================================================================

ABLATION_AXES = {
    "representation": ("representation", list(REPRESENTATIONS)),
    "timing": ("timing", [False, True]),
    "weights": ("weights", list(WEIGHT_SCHEMES)),
}


def ablation_base_candidate(fold_id):
    cfg = CONFIG["ablation"]
    head = cfg["base_family"]
    if head not in PIPELINE_BY_HEAD:
        raise ValueError("Ablação exige Ridge ou ExtraTrees.")
    candidate = {
        "family": head,
        "params": validate_model_params(head, cfg["base_params"]),
        "bin_hours": int(cfg["base_bin_hours"]),
        "horizon_days": float(cfg["base_horizon_days"]),
        "representation": cfg["base_representation"],
        "preprocessing": PREPROCESSING_CONFIG["method"],
        "add_missing_indicators": PREPROCESSING_CONFIG["add_missing_indicators"],
        "timing": bool(cfg["base_timing"]),
        "weights": cfg["base_weights"],
        "seed": seed_for(fold_id, head),
        "origin": "ablation",
        "fold_id": fold_id,
        "pipeline_id": PIPELINE_BY_HEAD[head],
    }
    candidate["candidate_id"] = timestral_candidate_id(candidate)
    return candidate


def ablation_levels():
    cfg = CONFIG["ablation"]
    if cfg["axis"] not in ABLATION_AXES:
        raise ValueError("Ablação: eixo deve ser representation, timing ou weights.")
    key, valid = ABLATION_AXES[cfg["axis"]]
    levels = list(cfg["levels"])
    if not levels or len(levels) != len(set(map(str, levels))):
        raise ValueError("Ablação: níveis vazios ou duplicados.")
    for level in levels:
        if level not in valid:
            raise ValueError(f"Ablação: nível inválido para {cfg['axis']}: {level!r}")
    return key, levels


def ablation_candidate_for_level(base, key, level, fold_id):
    candidate = json.loads(canonical_json(base))
    candidate[key] = level
    candidate["seed"] = seed_for(fold_id, base["family"])
    candidate["origin"] = f"ablation:{key}:{level}"
    candidate["candidate_id"] = timestral_candidate_id(candidate)
    return candidate


def run_ablation(get_features, counter):
    cfg = CONFIG["ablation"]
    if not cfg["enabled"]:
        return {}
    key, levels = ablation_levels()
    rows = []
    prediction_frames = []
    metric_rows = []
    for label in LABEL_ORDER:
        target = TARGETS[label]
        meta = PROTOCOL["labels"][label]
        for fold in SPLITS.to_dict(orient="records"):
            masks = masks_for_fold(target, fold)
            base = ablation_base_candidate(fold["fold_id"])
            for level in levels:
                candidate = ablation_candidate_for_level(
                    base, key, level, fold["fold_id"])
                test, prediction_scaled, _, fit_metadata = _selected_head_predict(
                    candidate, target, masks, get_features, counter,
                    purpose="ablation"
                )
                frame = predictions_frame(test, prediction_scaled, label,
                                          fold, candidate)
                frame["family"] = candidate["pipeline_id"]
                frame["head"] = candidate["family"]
                frame["ablation_axis"] = key
                frame["ablation_level"] = str(level)
                prediction_frames.append(frame)
                block = metric_breakdown(
                    frame["y_metric"].to_numpy(float),
                    frame["pred_metric"].to_numpy(float),
                    frame["Falha"].to_numpy(int),
                    frame["equipment"].to_numpy(str),
                    frame["phase"].to_numpy(str),
                    meta["unit"],
                )
                for scope, values in (("overall", {"all": block["overall"]}),
                                      ("component", block["by_component"]),
                                      ("equipment", block["by_equipment"]),
                                      ("phase", block["by_phase"])):
                    for name, metrics in values.items():
                        metric_rows.append({
                            "label": label,
                            "family": candidate["pipeline_id"],
                            "head": candidate["family"],
                            "unit": meta["unit"],
                            "fold_id": fold["fold_id"],
                            "axis": key,
                            "level": str(level),
                            "scope": scope,
                            "name": name,
                            **metrics,
                            "aggregation": "ablation_no_winner",
                        })
                rows.append({
                    "label": label,
                    "fold_id": fold["fold_id"],
                    "axis": key,
                    "level": str(level),
                    "candidate_id": candidate["candidate_id"],
                    "family": candidate["pipeline_id"],
                    "head": candidate["family"],
                    "horizon_days": candidate["horizon_days"],
                    "n_functional": fit_metadata.get("n_features_out", 0),
                })
    predictions = pd.concat(prediction_frames, ignore_index=True)
    metrics = pd.DataFrame(metric_rows)
    if predictions.empty or metrics.empty:
        raise RuntimeError("Ablação não produziu resultados.")
    ABLATION_DIR.mkdir(parents=True, exist_ok=True)
    write_csv(pd.DataFrame(rows), ABLATION_DIR / "ablation_fits.csv")
    write_csv(predictions, ABLATION_DIR / "ablation_predictions.csv")
    write_csv(metrics, ABLATION_DIR / "ablation_metrics.csv")
    summary = {
        "enabled": True,
        "family": cfg["base_family"],
        "axis": key,
        "levels": [str(level) for level in levels],
        "horizon_days": float(cfg["base_horizon_days"]),
        "n_predictions": int(len(predictions)),
        "n_levels": int(len(levels)),
        "no_outer_winner_selection": True,
        "metrics": metrics.to_dict(orient="records"),
    }
    atomic_write_json(ABLATION_DIR / "ablation_summary.json", summary)
    print("Ablação concluída:", key, levels, "| sem seleção de vencedor.")
    display(metrics.groupby(["family", "axis", "level", "scope"], sort=False)
            [["mae", "rmse", "bias"]].mean().reset_index())
    return summary


print("Ablação Timestral pronta (desabilitada por padrão).")


## Execução

A célula seguinte:

1. cria o diretório único da execução e exporta dados/auditoria/EDA/endpoints/splits;
2. imprime o **preview de forecasts** e de ajustes supervisionados **antes** de baixar pesos ou treinar;
3. no smoke de dados (`run_training=False`, `forecast_smoke=False`), não importa torch/Timestral, não baixa pesos e não ajusta heads; exporta o ZIP de dados;
4. no smoke explícito (`forecast_smoke=True`, `run_training=False`), executa o Timestral real em poucos endpoints, sem treinar heads;
5. no treino (`run_training=True` e `run_forecast=True`), calcula/reutiliza os forecasts congelados de todos os endpoints do dataset selecionado (gerador: 574; gearbox: 822), ajusta os dois heads por fold e exporta os payloads (gerador: 8; gearbox: 6).

Para treinar:

```python
CONFIG["run_training"] = True
CONFIG["model_specific"]["run_forecast"] = True
```

Depois **REEXECUTE TODAS AS CÉLULAS**. `RUN_ID`, o cache e as identidades incluem configuração/modelo; executar só a configuração e a última célula deixa a identidade obsoleta.

O forecast é sempre sensor-only. O teste externo não é usado para escolher grade, horizonte, representação, pré-processamento, pesos, parâmetros ou head.


In [ ]:
# =============================================================================
# SEÇÃO 18 — EXECUÇÃO: PREVIEW, FORECAST CONGELADO E HEADS
# =============================================================================

started_utc = RUNTIME_STARTED_UTC
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
export_data_tables()

FORECAST_PREVIEW = print_forecast_preview()
FIT_PREVIEW = estimate_expected_fits()
print_fit_preview(FIT_PREVIEW)

RUN_PAYLOADS = []
ABLATION_RESULTS = {}
FIT_COUNTER = {}
FORECAST_SMOKE_RESULT = None

if not CONFIG["run_training"]:
    if CONFIG["model_specific"]["forecast_smoke"]:
        FORECAST_SMOKE_RESULT = run_forecast_smoke()
    finished_utc = datetime.now(timezone.utc).isoformat()
    RESULTS_EXPORT = export_results(
        payloads=[], preview=FIT_PREVIEW, fit_counter=FIT_COUNTER,
        started_utc=started_utc, finished_utc=finished_utc,
        ablation_results={}, forecast_preview=FORECAST_PREVIEW,
    )
    if CONFIG["model_specific"]["forecast_smoke"]:
        print("\nSMOKE DE FORECAST CONCLUÍDO; nenhum head foi treinado.")
    else:
        print("\nSMOKE DE DADOS CONCLUÍDO; nenhum peso Timestral foi baixado "
              "e nenhum head foi treinado.")
    print("Para forecast real em todos os endpoints: "
          "model_specific.run_forecast=True e run_training=True.")
    print("Para smoke real curto: model_specific.forecast_smoke=True.")
    print("ZIP:", RESULTS_EXPORT["zip_path"])
else:
    if not CONFIG["model_specific"]["run_forecast"]:
        raise RuntimeError(
            "run_training=True exige model_specific.run_forecast=True; "
            "não há fallback silencioso nem mock em execução científica."
        )

    ensure_forecasts_for_plan()

    def get_features(width_hours, horizon_days):
        return get_timestral_feature_matrix(width_hours, horizon_days)

    for label in LABEL_ORDER:
        for fold in SPLITS.to_dict(orient="records"):
            for family in MODEL_FAMILIES:
                payload, reused = execute_label_fold_head(
                    label, fold, family, get_features, FIT_COUNTER
                )
                RUN_PAYLOADS.append(payload)

    if CONFIG["ablation"]["enabled"]:
        ABLATION_RESULTS = run_ablation(get_features, FIT_COUNTER)

    actual_total = int(FIT_COUNTER.get("model_fits", 0))
    actual_selection = int(FIT_COUNTER.get("selection_fits", 0))
    actual_final = int(FIT_COUNTER.get("final_fits", 0))
    if len(RUN_PAYLOADS) != expected_run_count():
        raise RuntimeError(
            f"Saídas incompletas: {len(RUN_PAYLOADS)} payloads, "
            f"esperado {expected_run_count()}.")
    if actual_total != int(FIT_PREVIEW["total_model_fits"]):
        raise RuntimeError(
            f"Contagem de ajustes divergente: preview={FIT_PREVIEW['total_model_fits']} "
            f"atual={actual_total}.")
    if actual_selection != int(FIT_PREVIEW["selection_fits"]):
        raise RuntimeError("Contagem de seleção divergente.")
    if actual_final != int(FIT_PREVIEW["final_fits"]):
        raise RuntimeError("Contagem de refits finais divergente.")
    if int(FORECAST_COUNTER.get("batches", 0)) != int(
            FORECAST_PREVIEW["expected_batches_missing"]):
        raise RuntimeError(
            "Contagem de batches de forecast divergente: "
            f"preview={FORECAST_PREVIEW['expected_batches_missing']} "
            f"atual={FORECAST_COUNTER.get('batches', 0)}."
        )

    finished_utc = datetime.now(timezone.utc).isoformat()
    RESULTS_EXPORT = export_results(
        payloads=RUN_PAYLOADS, preview=FIT_PREVIEW, fit_counter=FIT_COUNTER,
        started_utc=started_utc, finished_utc=finished_utc,
        ablation_results=ABLATION_RESULTS, forecast_preview=FORECAST_PREVIEW,
    )
    print("\nTreino concluído.")
    print("Payloads por família:", {
        family: sum(payload["family"] == family for payload in RUN_PAYLOADS)
        for family in MODEL_FAMILIES
    })
    print("Ajustes:", actual_total, "| seleção:", actual_selection,
          "| refit final:", actual_final)
    print("Forecast batches:", FORECAST_COUNTER.get("batches", 0),
          "| endpoints:", FORECAST_COUNTER.get("endpoints", 0))
    print("ZIP:", RESULTS_EXPORT["zip_path"])
    display(RESULTS_EXPORT["metrics_macro"].head(40))


## Leitura responsável

- O Timestral é congelado e prevê **sensores**, não RUL. A RUL só aparece no alvo dos heads Ridge/ExtraTrees.
- Não há RUL zero-shot, não há canal de RUL no contexto e não há covariável FPT conhecida no futuro.
- As incertezas dos sensores (`q10`/`q90`) não são intervalos de confiança da RUL. Nenhum limiar é interpretado como falha.
- Não há controle “sem forecast” nesta notebook. Assim, uma diferença em relação aos baselines standalone não isola o efeito do Timestral. A comparação opcional lê exports do outro notebook apenas para descrição, exige identidades compatíveis e nunca escolhe modelo pelo teste importado. Em smoke sem métricas Timestral, a comparação fica `deferred_no_current_metrics` e não é executada.
- Gerador: quatro equipamentos. Gearbox: três eventos, não quatro máquinas. Em ambos há um único grupo de validação interna por fold; a estimativa de seleção é de baixa resolução.
- O cenário é generalização para um grupo físico não visto, não previsão do futuro do mesmo equipamento ou evento.
- As datas FPT (gerador herdado: `1/2=2024-07-25`, `3=2024-03-15`, `4/5=2023-12-10`, `6/7=2024-07-22`; gearbox inferido: `E1=2023-06-14`, `E2=2022-11-01`, `E3=2023-08-13`) são anotações não verificadas fisicamente; só entram em relatório de fase e pesos opcionais de treino.
- SQLite em Drive não suporta múltiplos escritores com segurança; use um runtime por vez.
- A execução científica real no Colab não foi feita na criação deste notebook. A verificação local foi estrutural/sintética e não ajustou modelos nem executou os CSVs.


In [ ]:
# Reexecução explícita dos testes leves no runtime Colab.
# Não baixa pesos e não treina modelos.
COMMON_TEST_RESULTS_RERUN = run_self_tests()
TIMESTRAL_TEST_RESULTS_RERUN = run_timestral_self_tests()
print("Testes comuns:", len(COMMON_TEST_RESULTS_RERUN))
print("Testes Timestral:", len(TIMESTRAL_TEST_RESULTS_RERUN))
